In [1]:
import os

# Change to your project directory (update the path as needed)
os.chdir(r"E:\ET_Article_With_MM_SA\Data_For_plots")

# Verify the change
print(os.getcwd())

E:\ET_Article_With_MM_SA\Data_For_plots


In [1]:
import os

# Change to your project directory (update the path as needed)
os.chdir(r"E:\ET_Article_With_MM_SA\Future data ST")

# Verify the change
print(os.getcwd())

E:\ET_Article_With_MM_SA\Future data ST


In [9]:
# -*- coding: utf-8 -*-

import os
import xarray as xr
import pandas as pd
import numpy as np

DATA_DIR = ""

files = {
    "CWatM": os.path.join(DATA_DIR, "ET_CWatM_mm_day.nc"),
    "GLDAS": os.path.join(DATA_DIR, "GLDAS_ET_2000_2019.nc"),
    "SMAP": os.path.join(DATA_DIR, "SMAP_ET_2000_2019.nc"),
}

def inspect_netcdf(label, path):
    print("\n" + "="*80)
    print(f"DATASET: {label}")
    print(f"PATH: {path}")
    print("="*80)

    ds = xr.open_dataset(path)

    print("\n--- FULL DATASET INFO ---")
    print(ds)

    print("\n--- DIMENSIONS ---")
    print(dict(ds.dims))

    print("\n--- COORDINATES ---")
    for c in ds.coords:
        print(f"\nCoordinate: {c}")
        print(f"  dims: {ds[c].dims}")
        print(f"  shape: {ds[c].shape}")
        print(f"  dtype: {ds[c].dtype}")
        print(f"  attrs: {ds[c].attrs}")

        try:
            vals = ds[c].values
            if vals.size > 0:
                print(f"  first values: {vals.flatten()[:5]}")
                print(f"  last values : {vals.flatten()[-5:]}")
        except Exception as e:
            print(f"  could not print values: {e}")

    print("\n--- DATA VARIABLES ---")
    for v in ds.data_vars:
        da = ds[v]
        print(f"\nVariable: {v}")
        print(f"  dims: {da.dims}")
        print(f"  shape: {da.shape}")
        print(f"  dtype: {da.dtype}")
        print(f"  attrs: {da.attrs}")

        try:
            arr = da.values
            print(f"  min: {np.nanmin(arr)}")
            print(f"  max: {np.nanmax(arr)}")
            print(f"  mean: {np.nanmean(arr)}")
        except Exception as e:
            print(f"  could not calculate min/max/mean: {e}")

    print("\n--- POSSIBLE TIME VARIABLES ---")
    possible_time_names = [
        "time", "Time", "TIME",
        "date", "Date",
        "month", "Month",
        "year", "Year",
        "valid_time",
        "t"
    ]

    found_any = False

    for name in possible_time_names:
        if name in ds.variables:
            found_any = True
            print(f"\n{name}:")
            print(ds[name])
            try:
                vals = ds[name].values
                print(f"  first values: {vals.flatten()[:10]}")
                print(f"  last values : {vals.flatten()[-10:]}")
            except Exception as e:
                print(f"  could not print values: {e}")

    if not found_any:
        print("No obvious time variable found from common names.")

    print("\n--- GLOBAL ATTRIBUTES ---")
    print(ds.attrs)

    ds.close()


for label, path in files.items():
    inspect_netcdf(label, path)


DATASET: CWatM
PATH: ET_CWatM_mm_day.nc

--- FULL DATASET INFO ---
<xarray.Dataset> Size: 685kB
Dimensions:          (time: 468, lat: 13, lon: 14)
Coordinates:
  * time             (time) datetime64[ns] 4kB 1981-01-01 ... 2019-12-01
  * lat              (lat) float64 104B 36.75 36.25 35.75 ... 31.75 31.25 30.75
  * lon              (lon) float64 112B 45.25 45.75 46.25 ... 50.75 51.25 51.75
Data variables:
    ET_final_mm_day  (time, lat, lon) float64 681kB ...

--- DIMENSIONS ---
{'time': 468, 'lat': 13, 'lon': 14}

--- COORDINATES ---

Coordinate: lon
  dims: ('lon',)
  shape: (14,)
  dtype: float64
  attrs: {'standard_name': 'longitude', 'long_name': 'longitude', 'units': 'degrees_east', 'axis': 'X'}
  first values: [45.25 45.75 46.25 46.75 47.25]
  last values : [49.75 50.25 50.75 51.25 51.75]

Coordinate: lat
  dims: ('lat',)
  shape: (13,)
  dtype: float64
  attrs: {'standard_name': 'latitude', 'long_name': 'latitude', 'units': 'degrees_north', 'axis': 'Y'}
  first values: [36.75

In [22]:
# -*- coding: utf-8 -*-

"""
Comprehensive Gridded ET Comparison
CWatM vs GLDAS vs SMAP

Purpose:
    Compare evapotranspiration estimated from a large-scale water balance model
    with two satellite / reanalysis ET products.

Important:
    - Station observations are intentionally excluded.
    - Full available time series are plotted for each dataset.
    - Statistical metrics are calculated only over the common time period.
    - Spatial, temporal, seasonal, and grid-cell-based comparisons are produced.
    - NetCDF variable names are made safe, so '/' in units will not cause errors.

Inputs:
    data/ET_CWatM_mm_day.nc
    data/GLDAS_ET_2000_2019.nc
    data/SMAP_ET_2000_2019.nc

Outputs:
    outputs_gridded_ET_comparison/
"""

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr


# ============================================================
# 1. PATH SETTINGS
# ============================================================

BASE_DIR = os.getcwd()

DATA_DIR = os.path.join(BASE_DIR, "")

CWATM_FILE = os.path.join(DATA_DIR, "ET_CWatM_mm_day.nc")
GLDAS_FILE = os.path.join(DATA_DIR, "GLDAS_ET_2000_2019.nc")
SMAP_FILE = os.path.join(DATA_DIR, "SMAP_ET_2000_2019.nc")

OUT_DIR = os.path.join(BASE_DIR, "outputs_gridded_ET_comparison")

TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")
NC_DIR = os.path.join(OUT_DIR, "netcdf")

TS_DIR = os.path.join(FIG_DIR, "time_series")
SCATTER_DIR = os.path.join(FIG_DIR, "scatter")
MAP_DIR = os.path.join(FIG_DIR, "spatial_maps")
SEASONAL_DIR = os.path.join(FIG_DIR, "seasonal_cycle")
METRIC_BAR_DIR = os.path.join(FIG_DIR, "metric_bars")

for d in [
    OUT_DIR,
    TABLE_DIR,
    FIG_DIR,
    NC_DIR,
    TS_DIR,
    SCATTER_DIR,
    MAP_DIR,
    SEASONAL_DIR,
    METRIC_BAR_DIR,
]:
    os.makedirs(d, exist_ok=True)


# ============================================================
# 2. ANALYSIS SETTINGS
# ============================================================

# Main unit:
#   "mm/day"    -> no conversion; monthly values remain monthly mean daily ET
#   "mm/month"  -> multiply each monthly value by number of days in that month
TARGET_UNIT = "mm/day"

# ET negative values are physically problematic.
# SMAP may contain negative values because of retrieval noise.
MASK_NEGATIVE_ET = True

# If grids are not identical, GLDAS and SMAP will be interpolated to CWatM grid.
INTERPOLATE_TO_CWATM_GRID = True

# Minimum valid paired months for grid-cell temporal metrics.
MIN_VALID_MONTHS = 12


# ============================================================
# 3. PLOT SETTINGS
# ============================================================

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 150


# ============================================================
# 4. BASIC FUNCTIONS
# ============================================================

def unit_tag():
    return TARGET_UNIT.replace("/", "_")


def get_ylabel():
    return f"ET ({TARGET_UNIT})"


def get_label(name):
    return f"{name} ET ({TARGET_UNIT})"


def safe_nc_name(text):
    """
    Create NetCDF-safe variable name.
    NetCDF variable names cannot contain '/', spaces, or special characters.
    """

    text = str(text)
    text = text.replace("/", "_")
    text = text.replace(" ", "_")
    text = text.replace("-", "_")
    text = re.sub(r"[^\w]", "_", text)
    text = re.sub(r"_+", "_", text)
    text = text.strip("_")

    if len(text) == 0:
        text = "variable"

    return text


def clean_file_name(text):
    text = str(text).replace(" ", "_")
    text = re.sub(r"[^\w\-_\.]", "", text)
    return text


def ensure_month_start_index(series_or_df):
    obj = series_or_df.copy()
    obj.index = pd.to_datetime(obj.index)
    obj.index = obj.index.to_period("M").to_timestamp()
    obj = obj.sort_index()
    return obj


def days_in_month_from_index(index):
    idx = pd.to_datetime(index)
    return idx.days_in_month.astype(float)


# ============================================================
# 5. NETCDF READER FUNCTIONS
# ============================================================

def extract_dates_from_long_name(long_name_attr):
    """
    Extract dates from long_name attributes such as:
        0_Evap_tavg_2000_01
        57_land_evapotranspiration_flux_2019_12
    """

    if isinstance(long_name_attr, str):
        long_names = [long_name_attr]
    else:
        long_names = list(long_name_attr)

    dates = []

    for item in long_names:
        item = str(item)

        match = re.search(r"(\d{4})_(\d{2})", item)

        if match is None:
            raise ValueError(f"Could not extract date from long_name item: {item}")

        year = int(match.group(1))
        month = int(match.group(2))

        dates.append(pd.Timestamp(year=year, month=month, day=1))

    return pd.DatetimeIndex(dates)


def standardize_xy_to_latlon(da):
    """
    Standardize coordinate names:
        x / longitude -> lon
        y / latitude  -> lat
    """

    rename_dict = {}

    for name in list(da.dims) + list(da.coords):
        low = str(name).lower()

        if low in ["x", "longitude"]:
            rename_dict[name] = "lon"

        elif low in ["y", "latitude"]:
            rename_dict[name] = "lat"

    if rename_dict:
        da = da.rename(rename_dict)

    if "lat" in da.coords:
        da = da.sortby("lat")

    if "lon" in da.coords:
        da = da.sortby("lon")

    return da


def open_et_dataset(path, dataset_name):
    """
    Open ET NetCDF.

    Supports:
        1. Standard NetCDF with time, lat, lon
        2. Raster-stack NetCDF with band, y, x and dates in long_name
    """

    print("\n" + "=" * 80)
    print(f"Opening dataset: {dataset_name}")
    print(f"Path: {path}")
    print("=" * 80)

    ds = xr.open_dataset(path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if len(candidate_vars) == 0:
        raise ValueError(f"No ET variable found in {dataset_name}")

    if "ET_final_mm_day" in candidate_vars:
        var_name = "ET_final_mm_day"
    elif "__xarray_dataarray_variable__" in candidate_vars:
        var_name = "__xarray_dataarray_variable__"
    else:
        var_name = candidate_vars[0]

    da = ds[var_name]

    print(f"Selected variable: {var_name}")
    print(f"Original dims: {da.dims}")
    print(f"Original shape: {da.shape}")

    if "time" in da.dims:
        da["time"] = pd.to_datetime(da["time"].values)

    elif "band" in da.dims:
        long_name = da.attrs.get("long_name", None)

        if long_name is None:
            raise ValueError(
                f"{dataset_name} has band dimension but no long_name date information."
            )

        dates = extract_dates_from_long_name(long_name)

        if len(dates) != da.sizes["band"]:
            raise ValueError(
                f"{dataset_name}: number of extracted dates does not match band size."
            )

        da = da.rename({"band": "time"})
        da = da.assign_coords(time=dates)

    else:
        raise ValueError(
            f"No time or band dimension found in {dataset_name}. Dims: {da.dims}"
        )

    da = standardize_xy_to_latlon(da)

    for dim in ["time", "lat", "lon"]:
        if dim not in da.dims:
            raise ValueError(
                f"{dataset_name}: required dimension '{dim}' not found. Final dims: {da.dims}"
            )

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    if MASK_NEGATIVE_ET:
        da = da.where(da >= 0)

    da.name = dataset_name

    print(f"Final dims: {da.dims}")
    print(f"Final shape: {da.shape}")
    print(
        "Time range:",
        pd.to_datetime(da.time.values[0]).strftime("%Y-%m"),
        "to",
        pd.to_datetime(da.time.values[-1]).strftime("%Y-%m"),
    )
    print(f"Lat range: {float(da.lat.min()):.3f} to {float(da.lat.max()):.3f}")
    print(f"Lon range: {float(da.lon.min()):.3f} to {float(da.lon.max()):.3f}")
    print(f"Min: {float(da.min(skipna=True)):.4f}")
    print(f"Max: {float(da.max(skipna=True)):.4f}")
    print(f"Mean: {float(da.mean(skipna=True)):.4f}")

    return da


# ============================================================
# 6. UNIT HANDLING
# ============================================================

def convert_unit_if_needed(da):
    """
    Convert monthly mean daily ET to monthly total if requested.
    """

    da = da.copy()

    if TARGET_UNIT == "mm/day":
        da.attrs["unit"] = TARGET_UNIT
        return da

    elif TARGET_UNIT == "mm/month":
        days = xr.DataArray(
            pd.to_datetime(da.time.values).days_in_month.astype(float),
            dims=["time"],
            coords={"time": da.time}
        )

        out = da * days
        out.name = da.name
        out.attrs["unit"] = TARGET_UNIT
        return out

    else:
        raise ValueError("TARGET_UNIT must be either 'mm/day' or 'mm/month'.")


# ============================================================
# 7. GRID ALIGNMENT
# ============================================================

def align_to_reference_grid(reference_da, target_da, target_name):
    """
    Align target dataset to CWatM grid if needed.
    """

    same_lat = np.array_equal(reference_da.lat.values, target_da.lat.values)
    same_lon = np.array_equal(reference_da.lon.values, target_da.lon.values)

    if same_lat and same_lon:
        print(f"{target_name}: grid already matches CWatM.")
        return target_da

    if not INTERPOLATE_TO_CWATM_GRID:
        raise ValueError(
            f"{target_name} grid does not match CWatM grid. "
            "Set INTERPOLATE_TO_CWATM_GRID=True to interpolate."
        )

    print(f"{target_name}: interpolating to CWatM grid...")

    out = target_da.interp(
        lat=reference_da.lat,
        lon=reference_da.lon,
        method="linear"
    )

    out.name = target_name
    out.attrs["unit"] = TARGET_UNIT

    return out


# ============================================================
# 8. TIME FUNCTIONS
# ============================================================

def get_common_time(*arrays):
    """
    Return common time among all input DataArrays.
    """

    common = pd.DatetimeIndex(pd.to_datetime(arrays[0].time.values))

    for da in arrays[1:]:
        common = common.intersection(
            pd.DatetimeIndex(pd.to_datetime(da.time.values))
        )

    common = common.sort_values()

    return common


def subset_time(da, times):
    return da.sel(time=times)


# ============================================================
# 9. SERIES FUNCTIONS
# ============================================================

def area_mean_series(da):
    spatial_dims = [d for d in da.dims if d != "time"]
    s = da.mean(dim=spatial_dims, skipna=True).to_series()
    s.name = da.name
    s = ensure_month_start_index(s)
    return s


def area_std_series(da):
    spatial_dims = [d for d in da.dims if d != "time"]
    s = da.std(dim=spatial_dims, skipna=True).to_series()
    s.name = da.name
    s = ensure_month_start_index(s)
    return s


def area_min_series(da):
    spatial_dims = [d for d in da.dims if d != "time"]
    s = da.min(dim=spatial_dims, skipna=True).to_series()
    s.name = da.name
    s = ensure_month_start_index(s)
    return s


def area_max_series(da):
    spatial_dims = [d for d in da.dims if d != "time"]
    s = da.max(dim=spatial_dims, skipna=True).to_series()
    s.name = da.name
    s = ensure_month_start_index(s)
    return s


# ============================================================
# 10. METRIC FUNCTIONS
# ============================================================

def pairwise_metrics(reference, compared):
    """
    Metrics between two paired series.

    Bias = compared - reference
    """

    df = pd.concat([reference, compared], axis=1).dropna()
    df.columns = ["ref", "sim"]

    n = len(df)

    if n < 3:
        return {
            "n": n,
            "Pearson_r": np.nan,
            "Spearman_r": np.nan,
            "R2": np.nan,
            "Bias": np.nan,
            "MAE": np.nan,
            "RMSE": np.nan,
            "ubRMSE": np.nan,
            "NSE": np.nan,
            "KGE": np.nan,
            "Mean_reference": np.nan,
            "Mean_compared": np.nan,
            "Std_reference": np.nan,
            "Std_compared": np.nan,
        }

    ref = df["ref"].values
    sim = df["sim"].values

    diff = sim - ref

    bias = np.mean(diff)
    mae = np.mean(np.abs(diff))
    rmse = np.sqrt(np.mean(diff ** 2))
    ubrmse = np.sqrt(np.mean((diff - bias) ** 2))

    try:
        pearson_r = pearsonr(ref, sim)[0]
    except Exception:
        pearson_r = np.nan

    try:
        spearman_r = spearmanr(ref, sim)[0]
    except Exception:
        spearman_r = np.nan

    r2 = pearson_r ** 2 if np.isfinite(pearson_r) else np.nan

    denom = np.sum((ref - np.mean(ref)) ** 2)
    nse = np.nan if denom == 0 else 1 - np.sum((sim - ref) ** 2) / denom

    if np.std(ref) == 0 or np.mean(ref) == 0 or not np.isfinite(pearson_r):
        kge = np.nan
    else:
        alpha = np.std(sim) / np.std(ref)
        beta = np.mean(sim) / np.mean(ref)
        kge = 1 - np.sqrt((pearson_r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)

    return {
        "n": n,
        "Pearson_r": pearson_r,
        "Spearman_r": spearman_r,
        "R2": r2,
        "Bias": bias,
        "MAE": mae,
        "RMSE": rmse,
        "ubRMSE": ubrmse,
        "NSE": nse,
        "KGE": kge,
        "Mean_reference": np.mean(ref),
        "Mean_compared": np.mean(sim),
        "Std_reference": np.std(ref),
        "Std_compared": np.std(sim),
    }


def pairwise_metrics_from_arrays(ref_da, sim_da, reference_name, compared_name, comparison_type):
    """
    Flatten two DataArrays and calculate overall metrics.
    """

    ref = ref_da.values.flatten()
    sim = sim_da.values.flatten()

    mask = np.isfinite(ref) & np.isfinite(sim)

    ref_s = pd.Series(ref[mask], name=reference_name)
    sim_s = pd.Series(sim[mask], name=compared_name)

    metrics = pairwise_metrics(ref_s, sim_s)

    rec = {
        "comparison_type": comparison_type,
        "reference": reference_name,
        "compared": compared_name,
        "unit": TARGET_UNIT,
    }

    rec.update(metrics)

    return rec


def grid_cell_metrics(ref_da, sim_da, reference_name, compared_name):
    """
    Calculate temporal metrics for each grid cell over common period.
    """

    lat_values = ref_da.lat.values
    lon_values = ref_da.lon.values

    records = []

    metric_arrays = {
        "Pearson_r": np.full((len(lat_values), len(lon_values)), np.nan),
        "Spearman_r": np.full((len(lat_values), len(lon_values)), np.nan),
        "R2": np.full((len(lat_values), len(lon_values)), np.nan),
        "Bias": np.full((len(lat_values), len(lon_values)), np.nan),
        "MAE": np.full((len(lat_values), len(lon_values)), np.nan),
        "RMSE": np.full((len(lat_values), len(lon_values)), np.nan),
        "ubRMSE": np.full((len(lat_values), len(lon_values)), np.nan),
        "NSE": np.full((len(lat_values), len(lon_values)), np.nan),
        "KGE": np.full((len(lat_values), len(lon_values)), np.nan),
        "n": np.full((len(lat_values), len(lon_values)), np.nan),
    }

    for i, lat in enumerate(lat_values):
        for j, lon in enumerate(lon_values):

            ref_s = ref_da.sel(lat=lat, lon=lon).to_series()
            sim_s = sim_da.sel(lat=lat, lon=lon).to_series()

            df = pd.concat([ref_s, sim_s], axis=1).dropna()
            df.columns = ["ref", "sim"]

            if len(df) < MIN_VALID_MONTHS:
                continue

            metrics = pairwise_metrics(df["ref"], df["sim"])

            for m in metric_arrays:
                metric_arrays[m][i, j] = metrics[m]

            records.append({
                "lat": lat,
                "lon": lon,
                "reference": reference_name,
                "compared": compared_name,
                "unit": TARGET_UNIT,
                **metrics
            })

    metric_ds = xr.Dataset(
        {
            metric: (
                ("lat", "lon"),
                values
            )
            for metric, values in metric_arrays.items()
        },
        coords={
            "lat": lat_values,
            "lon": lon_values,
        }
    )

    metric_ds.attrs["reference"] = reference_name
    metric_ds.attrs["compared"] = compared_name
    metric_ds.attrs["unit"] = TARGET_UNIT
    metric_ds.attrs["min_valid_months"] = MIN_VALID_MONTHS

    metric_df = pd.DataFrame(records)

    return metric_ds, metric_df


def monthly_spatial_correlation(ref_da, sim_da, reference_name, compared_name):
    """
    For each month, calculate spatial correlation between two gridded fields.
    """

    records = []

    for t in ref_da.time.values:
        ref_field = ref_da.sel(time=t).values.flatten()
        sim_field = sim_da.sel(time=t).values.flatten()

        mask = np.isfinite(ref_field) & np.isfinite(sim_field)

        n = int(mask.sum())

        if n < 3:
            r = np.nan
            p = np.nan
        else:
            r, p = pearsonr(ref_field[mask], sim_field[mask])

        records.append({
            "time": pd.to_datetime(t),
            "reference": reference_name,
            "compared": compared_name,
            "n_grid_cells": n,
            "spatial_Pearson_r": r,
            "p_value": p,
            "unit": TARGET_UNIT,
        })

    return pd.DataFrame(records)


# ============================================================
# 11. PLOT FUNCTIONS
# ============================================================

def plot_time_series_full(area_df_full, out_png):
    """
    Plot full available area-mean time series.
    """

    plt.figure(figsize=(13, 5))

    for col in area_df_full.columns:
        plt.plot(area_df_full.index, area_df_full[col], linewidth=1.8, label=col)

    plt.title("Area-Mean ET Time Series - Full Available Period")
    plt.xlabel("Time")
    plt.ylabel(get_ylabel())
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def plot_time_series_common(area_df_common, out_png):
    """
    Plot area-mean time series over common period.
    """

    plt.figure(figsize=(13, 5))

    for col in area_df_common.columns:
        plt.plot(area_df_common.index, area_df_common[col], linewidth=1.8, label=col)

    start = area_df_common.index.min().strftime("%Y-%m")
    end = area_df_common.index.max().strftime("%Y-%m")

    plt.title(f"Area-Mean ET Time Series - Common Period ({start} to {end})")
    plt.xlabel("Time")
    plt.ylabel(get_ylabel())
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def plot_scatter(x, y, xlabel, ylabel, title, out_png):
    """
    Scatter plot with 1:1 line, regression line, and statistics.
    """

    df = pd.concat([x, y], axis=1).dropna()
    df.columns = ["x", "y"]

    if len(df) < 3:
        return

    xval = df["x"].values
    yval = df["y"].values

    metrics = pairwise_metrics(df["x"], df["y"])

    plt.figure(figsize=(6.5, 6.5))
    plt.scatter(xval, yval, alpha=0.70)

    minv = np.nanmin([xval.min(), yval.min()])
    maxv = np.nanmax([xval.max(), yval.max()])

    pad = (maxv - minv) * 0.05 if maxv > minv else 1
    minp = minv - pad
    maxp = maxv + pad

    plt.plot(
        [minp, maxp],
        [minp, maxp],
        linestyle="--",
        linewidth=1.5,
        label="1:1 line"
    )

    coef = np.polyfit(xval, yval, 1)
    fit = np.poly1d(coef)

    xx = np.linspace(minp, maxp, 100)

    plt.plot(
        xx,
        fit(xx),
        linewidth=1.5,
        label=f"Fit: y={coef[0]:.2f}x+{coef[1]:.2f}"
    )

    info = (
        f"n = {metrics['n']}\n"
        f"r = {metrics['Pearson_r']:.3f}\n"
        f"RMSE = {metrics['RMSE']:.3f}\n"
        f"Bias = {metrics['Bias']:.3f}"
    )

    plt.text(
        0.05,
        0.95,
        info,
        transform=plt.gca().transAxes,
        va="top",
        bbox=dict(boxstyle="round", alpha=0.18)
    )

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.xlim(minp, maxp)
    plt.ylim(minp, maxp)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def plot_gridded_scatter(ref_da, sim_da, reference_name, compared_name, out_png):
    """
    Scatter using all grid cells and all common months.
    """

    ref = ref_da.values.flatten()
    sim = sim_da.values.flatten()

    mask = np.isfinite(ref) & np.isfinite(sim)

    ref_s = pd.Series(ref[mask], name=reference_name)
    sim_s = pd.Series(sim[mask], name=compared_name)

    plot_scatter(
        ref_s,
        sim_s,
        xlabel=get_label(reference_name),
        ylabel=get_label(compared_name),
        title=f"All Grid Cells and Months: {reference_name} vs {compared_name}",
        out_png=out_png
    )


def plot_map(da2d, title, out_png, cmap=None, label=None):
    """
    Simple spatial map for a 2D lat-lon DataArray.
    """

    plt.figure(figsize=(7, 6))

    im = plt.pcolormesh(
        da2d.lon,
        da2d.lat,
        da2d,
        shading="auto",
        cmap=cmap
    )

    if label is None:
        label = da2d.attrs.get("unit", TARGET_UNIT)

    plt.colorbar(im, label=label)

    plt.title(title)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def plot_seasonal_cycle(seasonal_df, out_png):
    """
    Plot monthly climatological cycle.
    """

    plt.figure(figsize=(9, 5))

    for col in seasonal_df.columns:
        plt.plot(
            seasonal_df.index,
            seasonal_df[col],
            marker="o",
            linewidth=1.8,
            label=col
        )

    plt.xticks(range(1, 13))
    plt.title("Mean Seasonal Cycle of Area-Mean ET - Common Period")
    plt.xlabel("Month")
    plt.ylabel(get_ylabel())
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def plot_metric_bar(metrics_df, metric, out_png):
    """
    Bar chart for pairwise area-mean metrics.
    """

    df = metrics_df[metrics_df["comparison_type"] == "AreaMean_TimeSeries"].copy()

    if df.empty or metric not in df.columns:
        return

    df["pair"] = df["reference"] + " vs " + df["compared"]

    plt.figure(figsize=(8, 4.5))
    plt.bar(df["pair"], df[metric])
    plt.title(f"Area-Mean Pairwise {metric} - Common Period")
    plt.xlabel("Comparison Pair")
    plt.ylabel(metric)
    plt.grid(True, axis="y", alpha=0.3)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


# ============================================================
# 12. LOAD AND PREPARE DATA
# ============================================================

print("\nLoading ET datasets...")

cwatm = open_et_dataset(CWATM_FILE, "CWatM")
gldas = open_et_dataset(GLDAS_FILE, "GLDAS")
smap = open_et_dataset(SMAP_FILE, "SMAP")

print("\nApplying unit setting...")

cwatm = convert_unit_if_needed(cwatm)
gldas = convert_unit_if_needed(gldas)
smap = convert_unit_if_needed(smap)

cwatm.name = "CWatM"
gldas.name = "GLDAS"
smap.name = "SMAP"

print("\nAligning grids...")

gldas = align_to_reference_grid(cwatm, gldas, "GLDAS")
smap = align_to_reference_grid(cwatm, smap, "SMAP")

print("\nFinding common period...")

common_time = get_common_time(cwatm, gldas, smap)

if len(common_time) == 0:
    raise ValueError("No common time period found among CWatM, GLDAS, and SMAP.")

common_start = common_time.min().strftime("%Y-%m")
common_end = common_time.max().strftime("%Y-%m")

print(f"Common period: {common_start} to {common_end}")
print(f"Number of common months: {len(common_time)}")

cwatm_common = subset_time(cwatm, common_time)
gldas_common = subset_time(gldas, common_time)
smap_common = subset_time(smap, common_time)


# ============================================================
# 13. SAVE BASIC DATA SUMMARY
# ============================================================

print("\nSaving dataset summary...")

summary_records = []

for name, da in [
    ("CWatM", cwatm),
    ("GLDAS", gldas),
    ("SMAP", smap),
]:
    summary_records.append({
        "dataset": name,
        "start": pd.to_datetime(da.time.values[0]),
        "end": pd.to_datetime(da.time.values[-1]),
        "n_months": da.sizes["time"],
        "n_lat": da.sizes["lat"],
        "n_lon": da.sizes["lon"],
        "min": float(da.min(skipna=True)),
        "max": float(da.max(skipna=True)),
        "mean": float(da.mean(skipna=True)),
        "unit": TARGET_UNIT,
        "negative_mask_applied": MASK_NEGATIVE_ET,
    })

summary_df = pd.DataFrame(summary_records)

summary_csv = os.path.join(TABLE_DIR, f"dataset_summary_{unit_tag()}.csv")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 14. AREA-MEAN TIME SERIES
# ============================================================

print("\nCreating area-mean time series...")

area_full_df = pd.concat(
    [
        area_mean_series(cwatm),
        area_mean_series(gldas),
        area_mean_series(smap),
    ],
    axis=1
)

area_full_df.columns = ["CWatM", "GLDAS", "SMAP"]
area_full_df = area_full_df.sort_index()

area_common_df = area_full_df.loc[common_time, ["CWatM", "GLDAS", "SMAP"]]

area_full_csv = os.path.join(TABLE_DIR, f"area_mean_full_available_{unit_tag()}.csv")
area_common_csv = os.path.join(TABLE_DIR, f"area_mean_common_period_{unit_tag()}.csv")

area_full_df.to_csv(area_full_csv, encoding="utf-8-sig")
area_common_df.to_csv(area_common_csv, encoding="utf-8-sig")

plot_time_series_full(
    area_full_df,
    out_png=os.path.join(TS_DIR, f"area_mean_full_available_{unit_tag()}.png")
)

plot_time_series_common(
    area_common_df,
    out_png=os.path.join(TS_DIR, f"area_mean_common_period_{unit_tag()}.png")
)


# ============================================================
# 15. AREA STATISTICS TIME SERIES
# ============================================================

print("\nCreating area statistics time series...")

area_stats_records = []

for name, da in [
    ("CWatM", cwatm),
    ("GLDAS", gldas),
    ("SMAP", smap),
]:
    temp_df = pd.DataFrame({
        "mean": area_mean_series(da),
        "std": area_std_series(da),
        "min": area_min_series(da),
        "max": area_max_series(da),
    })

    temp_df["dataset"] = name
    temp_df["unit"] = TARGET_UNIT
    temp_df = temp_df.reset_index().rename(columns={"index": "time"})

    area_stats_records.append(temp_df)

area_stats_df = pd.concat(area_stats_records, axis=0)

area_stats_csv = os.path.join(TABLE_DIR, f"area_statistics_full_available_{unit_tag()}.csv")
area_stats_df.to_csv(area_stats_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 16. PAIR DEFINITIONS
# ============================================================

pairs = [
    ("CWatM", "GLDAS", cwatm_common, gldas_common),
    ("CWatM", "SMAP", cwatm_common, smap_common),
    ("GLDAS", "SMAP", gldas_common, smap_common),
]


# ============================================================
# 17. PAIRWISE METRICS - AREA MEAN
# ============================================================

print("\nCalculating area-mean pairwise metrics...")

all_metrics_records = []

for ref_name, sim_name, ref_da, sim_da in pairs:

    metrics = pairwise_metrics(
        area_common_df[ref_name],
        area_common_df[sim_name]
    )

    rec = {
        "comparison_type": "AreaMean_TimeSeries",
        "reference": ref_name,
        "compared": sim_name,
        "common_start": common_start,
        "common_end": common_end,
        "unit": TARGET_UNIT,
    }

    rec.update(metrics)
    all_metrics_records.append(rec)

    plot_scatter(
        area_common_df[ref_name],
        area_common_df[sim_name],
        xlabel=get_label(ref_name),
        ylabel=get_label(sim_name),
        title=f"Area-Mean Monthly ET: {ref_name} vs {sim_name}",
        out_png=os.path.join(
            SCATTER_DIR,
            f"scatter_area_mean_{ref_name}_vs_{sim_name}_{unit_tag()}.png"
        )
    )


# ============================================================
# 18. PAIRWISE METRICS - ALL GRID CELLS AND MONTHS
# ============================================================

print("\nCalculating gridded all-sample metrics...")

for ref_name, sim_name, ref_da, sim_da in pairs:

    rec = pairwise_metrics_from_arrays(
        ref_da,
        sim_da,
        reference_name=ref_name,
        compared_name=sim_name,
        comparison_type="AllGridCells_AllMonths"
    )

    rec["common_start"] = common_start
    rec["common_end"] = common_end

    all_metrics_records.append(rec)

    plot_gridded_scatter(
        ref_da,
        sim_da,
        reference_name=ref_name,
        compared_name=sim_name,
        out_png=os.path.join(
            SCATTER_DIR,
            f"scatter_all_grid_cells_all_months_{ref_name}_vs_{sim_name}_{unit_tag()}.png"
        )
    )

metrics_df = pd.DataFrame(all_metrics_records)

metrics_csv = os.path.join(TABLE_DIR, f"pairwise_metrics_common_period_{unit_tag()}.csv")
metrics_df.to_csv(metrics_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 19. GRID-CELL TEMPORAL METRICS
# ============================================================

print("\nCalculating grid-cell temporal metrics...")

grid_metric_tables = []
grid_metrics_all_csv = None

for ref_name, sim_name, ref_da, sim_da in pairs:

    metric_ds, metric_df = grid_cell_metrics(
        ref_da,
        sim_da,
        reference_name=ref_name,
        compared_name=sim_name
    )

    pair_tag = f"{ref_name}_vs_{sim_name}"

    metric_nc = os.path.join(
        NC_DIR,
        f"grid_cell_temporal_metrics_{pair_tag}_{unit_tag()}.nc"
    )

    metric_csv = os.path.join(
        TABLE_DIR,
        f"grid_cell_temporal_metrics_{pair_tag}_{unit_tag()}.csv"
    )

    metric_ds.to_netcdf(metric_nc)
    metric_df.to_csv(metric_csv, index=False, encoding="utf-8-sig")

    grid_metric_tables.append(metric_df)

    for metric_name in ["Pearson_r", "Bias", "RMSE", "MAE", "KGE"]:

        da_map = metric_ds[metric_name].copy()
        da_map.name = safe_nc_name(f"{metric_name}_{pair_tag}_{unit_tag()}")
        da_map.attrs["unit"] = TARGET_UNIT

        label = metric_name
        if metric_name in ["Bias", "RMSE", "MAE"]:
            label = f"{metric_name} ({TARGET_UNIT})"

        plot_map(
            da_map,
            title=f"{metric_name}: {ref_name} vs {sim_name}",
            out_png=os.path.join(
                MAP_DIR,
                f"map_{metric_name}_{pair_tag}_{unit_tag()}.png"
            ),
            label=label
        )

if len(grid_metric_tables) > 0:
    grid_metrics_all_df = pd.concat(grid_metric_tables, axis=0)

    grid_metrics_all_csv = os.path.join(
        TABLE_DIR,
        f"grid_cell_temporal_metrics_all_pairs_{unit_tag()}.csv"
    )

    grid_metrics_all_df.to_csv(grid_metrics_all_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 20. MEAN FIELD AND DIFFERENCE MAPS
# ============================================================

print("\nCreating mean ET and difference maps...")

mean_fields = {
    "CWatM": cwatm_common.mean(dim="time", skipna=True),
    "GLDAS": gldas_common.mean(dim="time", skipna=True),
    "SMAP": smap_common.mean(dim="time", skipna=True),
}

for name, field in mean_fields.items():

    field = field.copy()
    field.name = safe_nc_name(f"Mean_ET_{name}_{unit_tag()}")

    field.attrs["long_name"] = f"Mean ET - {name} - Common Period"
    field.attrs["unit"] = TARGET_UNIT
    field.attrs["common_start"] = common_start
    field.attrs["common_end"] = common_end

    mean_nc = os.path.join(
        NC_DIR,
        f"mean_ET_{name}_common_period_{unit_tag()}.nc"
    )

    field.to_netcdf(mean_nc)

    plot_map(
        field,
        title=f"Mean ET - {name} - Common Period ({TARGET_UNIT})",
        out_png=os.path.join(
            MAP_DIR,
            f"map_mean_ET_{name}_common_period_{unit_tag()}.png"
        ),
        label=get_ylabel()
    )


difference_records = []

for ref_name, sim_name, ref_da, sim_da in pairs:

    diff = sim_da.mean(dim="time", skipna=True) - ref_da.mean(dim="time", skipna=True)

    diff = diff.copy()
    diff.name = safe_nc_name(
        f"Mean_Difference_{sim_name}_minus_{ref_name}_{unit_tag()}"
    )

    diff.attrs["long_name"] = f"Mean Difference: {sim_name} minus {ref_name}"
    diff.attrs["unit"] = TARGET_UNIT
    diff.attrs["common_start"] = common_start
    diff.attrs["common_end"] = common_end

    diff_nc = os.path.join(
        NC_DIR,
        f"mean_difference_{sim_name}_minus_{ref_name}_{unit_tag()}.nc"
    )

    diff.to_netcdf(diff_nc)

    plot_map(
        diff,
        title=f"Mean Difference: {sim_name} - {ref_name} ({TARGET_UNIT})",
        out_png=os.path.join(
            MAP_DIR,
            f"map_mean_difference_{sim_name}_minus_{ref_name}_{unit_tag()}.png"
        ),
        label=f"Difference ({TARGET_UNIT})"
    )

    difference_records.append({
        "reference": ref_name,
        "compared": sim_name,
        "difference": f"{sim_name} - {ref_name}",
        "mean_difference": float(diff.mean(skipna=True)),
        "min_difference": float(diff.min(skipna=True)),
        "max_difference": float(diff.max(skipna=True)),
        "unit": TARGET_UNIT,
        "common_start": common_start,
        "common_end": common_end,
    })

difference_df = pd.DataFrame(difference_records)

difference_csv = os.path.join(
    TABLE_DIR,
    f"mean_difference_summary_{unit_tag()}.csv"
)

difference_df.to_csv(difference_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 21. MONTHLY SPATIAL CORRELATION
# ============================================================

print("\nCalculating monthly spatial correlations...")

spatial_corr_tables = []
spatial_corr_all_csv = None

for ref_name, sim_name, ref_da, sim_da in pairs:

    df_corr = monthly_spatial_correlation(
        ref_da,
        sim_da,
        reference_name=ref_name,
        compared_name=sim_name
    )

    spatial_corr_tables.append(df_corr)

    pair_tag = f"{ref_name}_vs_{sim_name}"

    out_csv = os.path.join(
        TABLE_DIR,
        f"monthly_spatial_correlation_{pair_tag}_{unit_tag()}.csv"
    )

    df_corr.to_csv(out_csv, index=False, encoding="utf-8-sig")

    plt.figure(figsize=(12, 4.5))
    plt.plot(
        df_corr["time"],
        df_corr["spatial_Pearson_r"],
        linewidth=1.8
    )
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.title(f"Monthly Spatial Correlation: {ref_name} vs {sim_name}")
    plt.xlabel("Time")
    plt.ylabel("Spatial Pearson r")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            TS_DIR,
            f"monthly_spatial_correlation_{pair_tag}_{unit_tag()}.png"
        ),
        dpi=300
    )
    plt.close()

if len(spatial_corr_tables) > 0:
    spatial_corr_all_df = pd.concat(spatial_corr_tables, axis=0)

    spatial_corr_all_csv = os.path.join(
        TABLE_DIR,
        f"monthly_spatial_correlation_all_pairs_{unit_tag()}.csv"
    )

    spatial_corr_all_df.to_csv(spatial_corr_all_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 22. SEASONAL CYCLE - COMMON PERIOD
# ============================================================

print("\nCreating seasonal cycle analysis...")

seasonal_df = area_common_df.copy()
seasonal_df["month"] = seasonal_df.index.month

seasonal_mean_df = seasonal_df.groupby("month")[["CWatM", "GLDAS", "SMAP"]].mean()

seasonal_csv = os.path.join(
    TABLE_DIR,
    f"seasonal_cycle_area_mean_common_period_{unit_tag()}.csv"
)

seasonal_mean_df.to_csv(seasonal_csv, encoding="utf-8-sig")

plot_seasonal_cycle(
    seasonal_mean_df,
    out_png=os.path.join(
        SEASONAL_DIR,
        f"seasonal_cycle_area_mean_common_period_{unit_tag()}.png"
    )
)


# ============================================================
# 23. SUMMARY METRIC BAR PLOTS
# ============================================================

print("\nCreating summary metric bar plots...")

for metric in ["Pearson_r", "R2", "Bias", "MAE", "RMSE", "ubRMSE", "KGE"]:
    plot_metric_bar(
        metrics_df,
        metric,
        out_png=os.path.join(
            METRIC_BAR_DIR,
            f"bar_area_mean_{metric}_{unit_tag()}.png"
        )
    )


# ============================================================
# 24. FINAL REPORT
# ============================================================

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print(f"\nTarget unit: {TARGET_UNIT}")
print(f"Negative ET masked: {MASK_NEGATIVE_ET}")
print(f"Common period: {common_start} to {common_end}")
print(f"Common months: {len(common_time)}")

print("\nMain output folder:")
print(OUT_DIR)

print("\nImportant output tables:")
print(summary_csv)
print(area_full_csv)
print(area_common_csv)
print(area_stats_csv)
print(metrics_csv)
print(difference_csv)
print(seasonal_csv)

if grid_metrics_all_csv is not None:
    print(grid_metrics_all_csv)

if spatial_corr_all_csv is not None:
    print(spatial_corr_all_csv)

print("\nMain figure folders:")
print(TS_DIR)
print(SCATTER_DIR)
print(MAP_DIR)
print(SEASONAL_DIR)
print(METRIC_BAR_DIR)

print("\nScientific note:")
print("Station observations were excluded because they are point-scale measurements,")
print("whereas CWatM, GLDAS, and SMAP represent gridded cell-scale ET estimates.")
print("All statistical comparisons were performed over the common time period only.")
print("Full available time series were plotted to show the temporal coverage of each dataset.")


Loading ET datasets...

Opening dataset: CWatM
Path: E:\ET_Article_With_MM_SA\Data_For_plots\ET_CWatM_mm_day.nc
Selected variable: ET_final_mm_day
Original dims: ('time', 'lat', 'lon')
Original shape: (468, 13, 14)
Final dims: ('time', 'lat', 'lon')
Final shape: (468, 13, 14)
Time range: 1981-01 to 2019-12
Lat range: 30.750 to 36.750
Lon range: 45.250 to 51.750
Min: 0.0003
Max: 6.3174
Mean: 1.1024

Opening dataset: GLDAS
Path: E:\ET_Article_With_MM_SA\Data_For_plots\GLDAS_ET_2000_2019.nc
Selected variable: __xarray_dataarray_variable__
Original dims: ('band', 'y', 'x')
Original shape: (240, 13, 14)
Final dims: ('time', 'lat', 'lon')
Final shape: (240, 13, 14)
Time range: 2000-01 to 2019-12
Lat range: 30.750 to 36.750
Lon range: 45.250 to 51.750
Min: 0.0020
Max: 3.9254
Mean: 0.7936

Opening dataset: SMAP
Path: E:\ET_Article_With_MM_SA\Data_For_plots\SMAP_ET_2000_2019.nc
Selected variable: __xarray_dataarray_variable__
Original dims: ('band', 'y', 'x')
Original shape: (58, 13, 14)
Final

In [3]:
# -*- coding: utf-8 -*-

import os
import xarray as xr
import pandas as pd
import numpy as np


# ============================================================
# 1. SET PATHS
# ============================================================

# مسیر اصلی داده‌های آینده را اینجا تنظیم کن
FUTURE_BASE_DIR = ""

# برای تست اولیه
TEST_MODEL = "GFDL"
TEST_SCENARIO = "126"

TEST_DIR = os.path.join(FUTURE_BASE_DIR, TEST_MODEL, TEST_SCENARIO)

FILES = {
    "totalET": os.path.join(TEST_DIR, "totalET_monthavg.nc"),
    "EvapoChannel": os.path.join(TEST_DIR, "EvapoChannel_monthavg.nc"),
    "EvapWaterBodyM": os.path.join(TEST_DIR, "EvapWaterBodyM_monthavg.nc"),
}


# ============================================================
# 2. INSPECTION FUNCTIONS
# ============================================================

def print_separator(title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def inspect_netcdf(label, path):
    print_separator(f"FILE: {label}")
    print(f"Path: {path}")

    if not os.path.exists(path):
        print("ERROR: File does not exist.")
        return None

    ds = xr.open_dataset(path)

    print("\n--- FULL DATASET INFO ---")
    print(ds)

    print("\n--- GLOBAL ATTRIBUTES ---")
    if len(ds.attrs) == 0:
        print("No global attributes.")
    else:
        for k, v in ds.attrs.items():
            print(f"{k}: {v}")

    print("\n--- DIMENSIONS ---")
    print(dict(ds.dims))

    print("\n--- COORDINATES ---")
    if len(ds.coords) == 0:
        print("No coordinates found.")
    else:
        for c in ds.coords:
            da = ds[c]
            print(f"\nCoordinate: {c}")
            print(f"  dims  : {da.dims}")
            print(f"  shape : {da.shape}")
            print(f"  dtype : {da.dtype}")
            print(f"  attrs : {da.attrs}")

            try:
                vals = da.values
                vals_flat = vals.flatten()
                print(f"  first values: {vals_flat[:8]}")
                print(f"  last values : {vals_flat[-8:]}")
            except Exception as e:
                print(f"  value preview error: {e}")

    print("\n--- DATA VARIABLES ---")
    if len(ds.data_vars) == 0:
        print("No data variables found.")
    else:
        for v in ds.data_vars:
            da = ds[v]

            print(f"\nVariable: {v}")
            print(f"  dims  : {da.dims}")
            print(f"  shape : {da.shape}")
            print(f"  dtype : {da.dtype}")
            print(f"  attrs : {da.attrs}")

            try:
                min_val = float(da.min(skipna=True).values)
                max_val = float(da.max(skipna=True).values)
                mean_val = float(da.mean(skipna=True).values)

                print(f"  min   : {min_val}")
                print(f"  max   : {max_val}")
                print(f"  mean  : {mean_val}")
            except Exception as e:
                print(f"  stats error: {e}")

    print("\n--- POSSIBLE TIME VARIABLES / COORDINATES ---")
    found_time_like = False

    for item in list(ds.coords) + list(ds.data_vars):
        low = str(item).lower()

        if any(k in low for k in ["time", "date", "month", "year", "day", "band"]):
            found_time_like = True
            print(f"\nTime-like item: {item}")
            print(ds[item])

            try:
                vals = ds[item].values.flatten()
                print(f"  first values: {vals[:10]}")
                print(f"  last values : {vals[-10:]}")
            except Exception:
                pass

    if not found_time_like:
        print("No obvious time-like variable found.")

    print("\n--- POSSIBLE AREA VARIABLES ---")
    found_area_like = False

    for item in list(ds.coords) + list(ds.data_vars):
        low = str(item).lower()

        if any(k in low for k in ["area", "cell", "gridcell", "m2", "m^2"]):
            found_area_like = True
            print(f"\nArea-like item: {item}")
            print(ds[item])
            try:
                vals = ds[item].values.flatten()
                print(f"  first values: {vals[:10]}")
                print(f"  last values : {vals[-10:]}")
            except Exception:
                pass

    if not found_area_like:
        print("No obvious area-like variable found.")

    return ds


def compare_three_files(datasets):
    print_separator("CROSS-FILE COMPARISON")

    valid = {k: v for k, v in datasets.items() if v is not None}

    if len(valid) < 2:
        print("Not enough files were opened for comparison.")
        return

    for name, ds in valid.items():
        print(f"\n{name}")
        print(f"  dims: {dict(ds.dims)}")
        print(f"  coords: {list(ds.coords)}")
        print(f"  data_vars: {list(ds.data_vars)}")

    print("\n--- VARIABLE CANDIDATES EXCLUDING spatial_ref/crs ---")
    for name, ds in valid.items():
        candidate_vars = [
            v for v in ds.data_vars
            if str(v).lower() not in ["spatial_ref", "crs"]
        ]
        print(f"{name}: {candidate_vars}")

    print("\n--- COORDINATE RANGES ---")
    for name, ds in valid.items():
        for lat_name in ["lat", "latitude", "y"]:
            if lat_name in ds.coords:
                lat = ds[lat_name]
                print(f"{name} {lat_name}: {float(lat.min()):.6f} to {float(lat.max()):.6f}")
                break

        for lon_name in ["lon", "longitude", "x"]:
            if lon_name in ds.coords:
                lon = ds[lon_name]
                print(f"{name} {lon_name}: {float(lon.min()):.6f} to {float(lon.max()):.6f}")
                break


# ============================================================
# 3. RUN INSPECTION
# ============================================================

opened = {}

for label, path in FILES.items():
    opened[label] = inspect_netcdf(label, path)

compare_three_files(opened)

print("\nDone. Please copy the printed output here.")


FILE: totalET
Path: GFDL\126\totalET_monthavg.nc

--- FULL DATASET INFO ---
<xarray.Dataset> Size: 3MB
Dimensions:           (time: 1032, lat: 27, lon: 30)
Coordinates:
  * time              (time) datetime64[ns] 8kB 2015-01-01 ... 2100-12-01
  * lat               (lat) float64 216B 39.75 39.25 38.75 ... 27.75 27.25 26.75
  * lon               (lon) float64 240B 37.25 37.75 38.25 ... 50.75 51.25 51.75
Data variables:
    totalET_monthavg  (time, lat, lon) float32 3MB ...
Attributes:
    settingsfile:     D:\Mehdi\mehdi\CWatM\Toolkit\Calibration\runs_calibrati...
    run_created:      Wed Sep 17 10:23:47 2025
    Source_Software:  CWATM Python: D:\Mehdi\mehdi\CWatM\cwatm\run_cwatm.py
    Platform:         Windows
    Version:          1.5: run_cwatm.py 2025/04/27 15:58
    institution:      IIASA
    title:            Rhine Water Model - WATCH WDFEI
    source:           CWATM output maps
    Conventions:      CF-1.6
    description:      Total evapotranspiration for each cell includin

C:\Users\AAAli\AppData\Local\Temp\ipykernel_20584\2651212641.py:60: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(dict(ds.dims))



--- FULL DATASET INFO ---
<xarray.Dataset> Size: 3MB
Dimensions:                (time: 1032, lat: 27, lon: 30)
Coordinates:
  * time                   (time) datetime64[ns] 8kB 2015-01-01 ... 2100-12-01
  * lat                    (lat) float64 216B 39.75 39.25 38.75 ... 27.25 26.75
  * lon                    (lon) float64 240B 37.25 37.75 38.25 ... 51.25 51.75
Data variables:
    EvapoChannel_monthavg  (time, lat, lon) float32 3MB ...
Attributes:
    settingsfile:     D:\Mehdi\mehdi\CWatM\Toolkit\Calibration\runs_calibrati...
    run_created:      Wed Sep 17 10:23:47 2025
    Source_Software:  CWATM Python: D:\Mehdi\mehdi\CWatM\cwatm\run_cwatm.py
    Platform:         Windows
    Version:          1.5: run_cwatm.py 2025/04/27 15:58
    institution:      IIASA
    title:            Rhine Water Model - WATCH WDFEI
    source:           CWATM output maps
    Conventions:      CF-1.6
    description:      Channel evaporation
    author:           IIASA WAT

--- GLOBAL ATTRIBUTES ---
setti

C:\Users\AAAli\AppData\Local\Temp\ipykernel_20584\2651212641.py:60: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(dict(ds.dims))


  min   : 1.2235670179263166e-16
  max   : 0.00017772131832316518
  mean  : 6.8149311118759215e-06

--- POSSIBLE TIME VARIABLES / COORDINATES ---

Time-like item: time
<xarray.DataArray 'time' (time: 1032)> Size: 8kB
array(['2015-01-01T00:00:00.000000000', '2015-02-01T00:00:00.000000000',
       '2015-03-01T00:00:00.000000000', ..., '2100-10-01T00:00:00.000000000',
       '2100-11-01T00:00:00.000000000', '2100-12-01T00:00:00.000000000'],
      dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 8kB 2015-01-01 2015-02-01 ... 2100-12-01
Attributes:
    standard_name:  time
  first values: ['2015-01-01T00:00:00.000000000' '2015-02-01T00:00:00.000000000'
 '2015-03-01T00:00:00.000000000' '2015-04-01T00:00:00.000000000'
 '2015-05-01T00:00:00.000000000' '2015-06-01T00:00:00.000000000'
 '2015-07-01T00:00:00.000000000' '2015-08-01T00:00:00.000000000'
 '2015-09-01T00:00:00.000000000' '2015-10-01T00:00:00.000000000']
  last values : ['2100-03-01T00:00:00.000000000' '2100-04-01

C:\Users\AAAli\AppData\Local\Temp\ipykernel_20584\2651212641.py:60: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(dict(ds.dims))
C:\Users\AAAli\AppData\Local\Temp\ipykernel_20584\2651212641.py:161: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  dims: {dict(ds.dims)}")


In [ ]:
# -*- coding: utf-8 -*-

"""
Historical-Future ET Time Series
Five Climate Models and Three Scenarios

Historical:
    CWatM
    GLDAS
    SMAP

Future climate models:
    GFDL
    IPSL
    MPI
    MRI
    UKESM

Scenarios:
    126
    370
    585

Future mode:
    Conservative mode:
        Future_ET = totalET_monthavg only

Reason:
    totalET_monthavg is already total evapotranspiration.
    EvapoChannel and EvapWaterBodyM are not added in this mode to avoid double counting.

Main outputs:
    Three figures:
        historical_future_ET_scenario_126_totalET_only_5models.png
        historical_future_ET_scenario_370_totalET_only_5models.png
        historical_future_ET_scenario_585_totalET_only_5models.png
"""

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


# ============================================================
# 1. PATH SETTINGS
# ============================================================

# مسیر اصلی همان پوشه‌ای است که فایل‌های nc تاریخی و پوشه‌های GFDL, IPSL, MPI, MRI, UKESM داخل آن هستند.
BASE_DIR = os.getcwd()

# Historical files are directly inside BASE_DIR according to your screenshot
CWATM_FILE = os.path.join(BASE_DIR, "ET_CWatM_mm_day.nc")
GLDAS_FILE = os.path.join(BASE_DIR, "GLDAS_ET_2000_2019.nc")
SMAP_FILE = os.path.join(BASE_DIR, "SMAP_ET_2000_2019.nc")

# Future model folders are also directly inside BASE_DIR
FUTURE_BASE_DIR = BASE_DIR


# ============================================================
# 2. MODELS AND SCENARIOS
# ============================================================

CLIMATE_MODELS = [
    "GFDL",
    "IPSL",
    "MPI",
    "MRI",
    "UKESM",
]

SCENARIOS = {
    "126": "Optimistic / SSP126",
    "370": "Medium / SSP370",
    "585": "Pessimistic / SSP585",
}

FUTURE_TOTAL_ET_FILE = "totalET_monthavg.nc"
FUTURE_TOTAL_ET_VAR = "totalET_monthavg"


# ============================================================
# 3. OUTPUT SETTINGS
# ============================================================

OUT_DIR = os.path.join(BASE_DIR, "outputs_historical_future_ET_totalET_only_5models")
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, TABLE_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)


# ============================================================
# 4. ANALYSIS SETTINGS
# ============================================================

FUTURE_ET_MODE = "totalET_only"

# Historical products are assumed to be monthly mean daily ET, mm/day
HISTORICAL_UNIT = "mm/day"

# Future totalET_monthavg has unit m in the files.
# Multiplying by 1000 gives mm-scale.
FUTURE_CONVERT_M_TO_MM = True

MASK_NEGATIVE_HISTORICAL_ET = True
MASK_NEGATIVE_FUTURE_ET = True

# Bias correction:
# True  = future models are adjusted to CWatM over the overlap period, usually 2015-2019.
# False = raw future totalET values are plotted.
APPLY_BIAS_CORRECTION = True
BIAS_REFERENCE = "CWatM"

# Multiplicative correction:
# corrected_future = raw_future * mean(CWatM_overlap) / mean(future_overlap)
BIAS_METHOD = "multiplicative_mean_ratio"

# 1  = raw monthly series
# 12 = 12-month rolling mean for smoother plots
ROLLING_MONTHS = 12

SAVE_ANNUAL_SERIES = True


# ============================================================
# 5. PLOT SETTINGS
# ============================================================

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 150


# ============================================================
# 6. BASIC FUNCTIONS
# ============================================================

def ensure_month_start_index(series_or_df):
    obj = series_or_df.copy()
    obj.index = pd.to_datetime(obj.index)
    obj.index = obj.index.to_period("M").to_timestamp()
    obj = obj.sort_index()
    return obj


def apply_rolling(series, months):
    if months is None or months <= 1:
        return series

    return series.rolling(
        window=months,
        center=True,
        min_periods=max(2, months // 2)
    ).mean()


def get_y_label():
    if APPLY_BIAS_CORRECTION:
        return "ET (bias-corrected; comparable historical-future scale)"
    return "ET (historical: mm/day; future totalET converted from m to mm)"


def save_series_annual(df, out_csv):
    annual_df = df.copy()
    annual_df.index = pd.to_datetime(annual_df.index)
    annual_df = annual_df.resample("YE").mean()
    annual_df.index = annual_df.index.year
    annual_df.index.name = "Year"
    annual_df.to_csv(out_csv, encoding="utf-8-sig")
    return annual_df


# ============================================================
# 7. HISTORICAL DATA READER
# ============================================================

def extract_dates_from_long_name(long_name_attr):
    """
    Extract dates from GLDAS/SMAP long_name attributes:
        0_Evap_tavg_2000_01
        57_land_evapotranspiration_flux_2019_12
    """

    if isinstance(long_name_attr, str):
        long_names = [long_name_attr]
    else:
        long_names = list(long_name_attr)

    dates = []

    for item in long_names:
        item = str(item)
        match = re.search(r"(\d{4})_(\d{2})", item)

        if match is None:
            raise ValueError(f"Cannot extract date from long_name item: {item}")

        dates.append(
            pd.Timestamp(
                year=int(match.group(1)),
                month=int(match.group(2)),
                day=1
            )
        )

    return pd.DatetimeIndex(dates)


def standardize_xy_to_latlon(da):
    """
    Rename coordinates:
        x / longitude -> lon
        y / latitude  -> lat
    """

    rename_dict = {}

    for name in list(da.dims) + list(da.coords):
        low = str(name).lower()

        if low in ["x", "longitude"]:
            rename_dict[name] = "lon"

        elif low in ["y", "latitude"]:
            rename_dict[name] = "lat"

    if rename_dict:
        da = da.rename(rename_dict)

    if "lat" in da.coords:
        da = da.sortby("lat")

    if "lon" in da.coords:
        da = da.sortby("lon")

    return da


def open_historical_et(path, dataset_name):
    print("\n" + "=" * 80)
    print(f"Opening historical dataset: {dataset_name}")
    print(path)
    print("=" * 80)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    ds = xr.open_dataset(path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if len(candidate_vars) == 0:
        raise ValueError(f"No variable found in {dataset_name}")

    if "ET_final_mm_day" in candidate_vars:
        var_name = "ET_final_mm_day"
    elif "__xarray_dataarray_variable__" in candidate_vars:
        var_name = "__xarray_dataarray_variable__"
    else:
        var_name = candidate_vars[0]

    da = ds[var_name]

    if "time" in da.dims:
        da["time"] = pd.to_datetime(da.time.values)

    elif "band" in da.dims:
        long_name = da.attrs.get("long_name", None)

        if long_name is None:
            raise ValueError(f"{dataset_name} has band dimension but no long_name.")

        dates = extract_dates_from_long_name(long_name)

        if len(dates) != da.sizes["band"]:
            raise ValueError(
                f"{dataset_name}: extracted dates do not match band size."
            )

        da = da.rename({"band": "time"})
        da = da.assign_coords(time=dates)

    else:
        raise ValueError(f"No time or band dimension found in {dataset_name}")

    da = standardize_xy_to_latlon(da)

    for d in ["time", "lat", "lon"]:
        if d not in da.dims:
            raise ValueError(f"{dataset_name}: required dimension '{d}' not found.")

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    if MASK_NEGATIVE_HISTORICAL_ET:
        da = da.where(da >= 0)

    da.name = dataset_name
    da.attrs["unit"] = HISTORICAL_UNIT

    print(f"Variable: {var_name}")
    print(f"Dims: {da.dims}")
    print(f"Shape: {da.shape}")
    print(
        "Time:",
        pd.to_datetime(da.time.values[0]).strftime("%Y-%m"),
        "to",
        pd.to_datetime(da.time.values[-1]).strftime("%Y-%m")
    )
    print(f"Lat: {float(da.lat.min()):.3f} to {float(da.lat.max()):.3f}")
    print(f"Lon: {float(da.lon.min()):.3f} to {float(da.lon.max()):.3f}")
    print(f"Mean: {float(da.mean(skipna=True)):.6f}")

    return da


# ============================================================
# 8. CELL AREA AND AREA-WEIGHTED MEAN
# ============================================================

def infer_bounds_from_centers(values):
    values = np.asarray(values, dtype=float)

    if len(values) < 2:
        raise ValueError("At least two coordinate values are needed.")

    values_sorted = np.sort(values)

    mid = (values_sorted[:-1] + values_sorted[1:]) / 2.0

    first = values_sorted[0] - (mid[0] - values_sorted[0])
    last = values_sorted[-1] + (values_sorted[-1] - mid[-1])

    return np.concatenate([[first], mid, [last]])


def calculate_latlon_cell_area(lat, lon, radius=6371000.0):
    """
    Calculate grid-cell area in m2 for regular lat-lon grid.
    """

    lat_values = np.asarray(lat.values if hasattr(lat, "values") else lat, dtype=float)
    lon_values = np.asarray(lon.values if hasattr(lon, "values") else lon, dtype=float)

    lat_ascending = np.all(np.diff(lat_values) > 0)
    lon_ascending = np.all(np.diff(lon_values) > 0)

    lat_sorted = np.sort(lat_values)
    lon_sorted = np.sort(lon_values)

    lat_bounds = infer_bounds_from_centers(lat_sorted)
    lon_bounds = infer_bounds_from_centers(lon_sorted)

    lat_bounds_rad = np.deg2rad(lat_bounds)
    lon_bounds_rad = np.deg2rad(lon_bounds)

    dlon = np.diff(lon_bounds_rad)
    sin_lat_diff = np.diff(np.sin(lat_bounds_rad))

    area_sorted = (radius ** 2) * sin_lat_diff[:, None] * dlon[None, :]
    area_sorted = np.abs(area_sorted)

    if not lat_ascending:
        area_sorted = area_sorted[::-1, :]

    if not lon_ascending:
        area_sorted = area_sorted[:, ::-1]

    area = xr.DataArray(
        area_sorted,
        dims=("lat", "lon"),
        coords={"lat": lat_values, "lon": lon_values},
        name="cell_area"
    )

    area.attrs["units"] = "m2"

    return area


def area_weighted_mean(da):
    """
    Area-weighted spatial mean over lat/lon.
    """

    area = calculate_latlon_cell_area(da.lat, da.lon)
    valid = da.notnull()

    weighted_sum = (da * area).where(valid).sum(dim=("lat", "lon"), skipna=True)
    weight_sum = area.where(valid).sum(dim=("lat", "lon"), skipna=True)

    out = weighted_sum / weight_sum
    out.name = da.name

    return out


def to_monthly_area_series(da):
    s = area_weighted_mean(da).to_series()
    s = ensure_month_start_index(s)
    return s


# ============================================================
# 9. DOMAIN FUNCTIONS
# ============================================================

def get_historical_domain(*arrays):
    """
    Use intersection of historical product bounding boxes.
    """

    lat_min = max([float(da.lat.min()) for da in arrays])
    lat_max = min([float(da.lat.max()) for da in arrays])
    lon_min = max([float(da.lon.min()) for da in arrays])
    lon_max = min([float(da.lon.max()) for da in arrays])

    return lat_min, lat_max, lon_min, lon_max


def clip_to_domain(da, domain):
    lat_min, lat_max, lon_min, lon_max = domain

    da = da.sortby("lat")
    da = da.sortby("lon")

    clipped = da.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )

    return clipped


# ============================================================
# 10. FUTURE TOTAL ET READER
# ============================================================

def open_future_total_et(model_name, scenario, hist_domain):
    """
    Open future totalET_monthavg.nc for one model and one scenario.
    Conservative mode:
        Future_ET = totalET_monthavg only
    """

    scen_dir = os.path.join(FUTURE_BASE_DIR, model_name, scenario)
    total_path = os.path.join(scen_dir, FUTURE_TOTAL_ET_FILE)

    if not os.path.exists(total_path):
        raise FileNotFoundError(total_path)

    ds = xr.open_dataset(total_path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if FUTURE_TOTAL_ET_VAR in candidate_vars:
        var_name = FUTURE_TOTAL_ET_VAR
    elif len(candidate_vars) == 1:
        var_name = candidate_vars[0]
    else:
        raise ValueError(
            f"Cannot select future total ET variable from {total_path}. "
            f"Candidates: {candidate_vars}"
        )

    da = ds[var_name]

    if "time" not in da.dims:
        raise ValueError(f"No time dimension found in {total_path}")

    da["time"] = pd.to_datetime(da.time.values)

    da = standardize_xy_to_latlon(da)

    for d in ["time", "lat", "lon"]:
        if d not in da.dims:
            raise ValueError(f"{model_name}/{scenario}: dimension '{d}' not found.")

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    da = clip_to_domain(da, hist_domain)

    if FUTURE_CONVERT_M_TO_MM:
        da = da * 1000.0

    if MASK_NEGATIVE_FUTURE_ET:
        da = da.where(da >= 0)

    da.name = f"{model_name}_{scenario}_totalET"

    return da


# ============================================================
# 11. BIAS CORRECTION
# ============================================================

def bias_correct_future_series(future_series, reference_series):
    """
    Multiplicative mean-ratio correction using overlap period.

    corrected = future * mean(reference_overlap) / mean(future_overlap)
    """

    df = pd.concat([future_series, reference_series], axis=1).dropna()
    df.columns = ["future", "reference"]

    n_overlap = len(df)

    if n_overlap < 12:
        return future_series, np.nan, n_overlap

    future_mean = df["future"].mean()
    reference_mean = df["reference"].mean()

    if not np.isfinite(future_mean) or future_mean == 0:
        return future_series, np.nan, n_overlap

    factor = reference_mean / future_mean

    corrected = future_series * factor
    corrected.name = future_series.name

    return corrected, factor, n_overlap


# ============================================================
# 12. FUTURE PERIODS
# ============================================================

def get_future_start_date_from_historical(historical_df):
    """
    Future starts from first month after the last available historical month.
    """

    last_hist_date = historical_df.dropna(how="all").index.max()
    future_start = pd.Timestamp(last_hist_date) + pd.DateOffset(months=1)
    future_start = future_start.to_period("M").to_timestamp()

    return future_start


def make_future_period_lines_and_labels(future_start, future_end):
    """
    Three 25-year future periods starting from future_start.
    """

    start1 = pd.Timestamp(future_start)
    start2 = start1 + pd.DateOffset(years=25)
    start3 = start2 + pd.DateOffset(years=25)
    start4 = start3 + pd.DateOffset(years=25)

    future_end = pd.Timestamp(future_end)

    if start4 > future_end:
        start4 = future_end

    lines = [start1, start2, start3, start4]

    labels = [
        ("Near future", start1, start2 - pd.DateOffset(months=1)),
        ("Mid-century", start2, start3 - pd.DateOffset(months=1)),
        ("Late century", start3, future_end),
    ]

    return lines, labels


# ============================================================
# 13. PLOTTING
# ============================================================

def plot_scenario_timeseries(
    scenario,
    scenario_label,
    historical_df,
    future_df,
    ensemble_series,
    future_period_lines,
    future_period_labels,
    out_png
):
    """
    Plot one scenario:
        Historical: CWatM, GLDAS, SMAP
        Future: 5 individual model lines + ensemble mean
    """

    plt.figure(figsize=(16, 6.5))

    hist_colors = {
        "CWatM": "black",
        "GLDAS": "tab:blue",
        "SMAP": "tab:green",
    }

    # Historical lines
    for col in historical_df.columns:
        s = apply_rolling(historical_df[col], ROLLING_MONTHS)

        plt.plot(
            s.index,
            s.values,
            linewidth=2.1,
            label=f"Historical {col}",
            color=hist_colors.get(col, None),
            zorder=3
        )

    # Future individual model lines
    model_colors = {
        "GFDL": "tab:blue",
        "IPSL": "tab:orange",
        "MPI": "tab:green",
        "MRI": "tab:purple",
        "UKESM": "tab:brown",
    }

    for col in future_df.columns:
        s = apply_rolling(future_df[col], ROLLING_MONTHS)

        plt.plot(
            s.index,
            s.values,
            linewidth=1.5,
            alpha=0.45,
            color=model_colors.get(col, None),
            label=f"Future {col}",
            zorder=4
        )

    # Ensemble mean
    ens = apply_rolling(ensemble_series, ROLLING_MONTHS)

    plt.plot(
        ens.index,
        ens.values,
        linewidth=3.4,
        color="red",
        alpha=0.95,
        label="Future ensemble mean",
        zorder=5
    )

    # Period lines
    for dt in future_period_lines:
        plt.axvline(
            pd.Timestamp(dt),
            color="gray",
            linestyle="--",
            linewidth=1.0,
            alpha=0.45,
            zorder=1
        )

    ymin, ymax = plt.ylim()

    # Period labels
    for label, start, end in future_period_labels:
        mid = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2

        plt.text(
            mid,
            ymax - 0.03 * (ymax - ymin),
            label,
            ha="center",
            va="top",
            fontsize=9,
            color="gray"
        )

    title_extra = "bias corrected to historical CWatM" if APPLY_BIAS_CORRECTION else "raw future"

    plt.title(
        f"Historical and Future Monthly ET Time Series - {scenario_label}\n"
        f"Future mode: totalET only; {title_extra}"
    )

    plt.xlabel("Time")
    plt.ylabel(get_y_label())
    plt.grid(True, alpha=0.25)
    plt.legend(ncol=3, fontsize=8)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


# ============================================================
# 14. MAIN WORKFLOW
# ============================================================

print("\nLoading historical ET datasets...")

cwatm = open_historical_et(CWATM_FILE, "CWatM")
gldas = open_historical_et(GLDAS_FILE, "GLDAS")
smap = open_historical_et(SMAP_FILE, "SMAP")

hist_domain = get_historical_domain(cwatm, gldas, smap)

print("\nHistorical domain intersection:")
print(f"lat: {hist_domain[0]:.3f} to {hist_domain[1]:.3f}")
print(f"lon: {hist_domain[2]:.3f} to {hist_domain[3]:.3f}")

# Clip historical data to common historical domain
cwatm_clip = clip_to_domain(cwatm, hist_domain)
gldas_clip = clip_to_domain(gldas, hist_domain)
smap_clip = clip_to_domain(smap, hist_domain)

historical_df = pd.concat(
    [
        to_monthly_area_series(cwatm_clip).rename("CWatM"),
        to_monthly_area_series(gldas_clip).rename("GLDAS"),
        to_monthly_area_series(smap_clip).rename("SMAP"),
    ],
    axis=1
)

historical_df = historical_df.sort_index()

historical_csv = os.path.join(TABLE_DIR, "historical_area_mean_ET.csv")
historical_df.to_csv(historical_csv, encoding="utf-8-sig")

if SAVE_ANNUAL_SERIES:
    save_series_annual(
        historical_df,
        os.path.join(TABLE_DIR, "historical_area_mean_ET_annual.csv")
    )

future_start_date = get_future_start_date_from_historical(historical_df)
future_end_date = pd.Timestamp("2100-12-01")

future_period_lines, future_period_labels = make_future_period_lines_and_labels(
    future_start=future_start_date,
    future_end=future_end_date
)

print("\nFuture plotting starts from:")
print(future_start_date.strftime("%Y-%m"))

print("\nClimate models:")
print(CLIMATE_MODELS)


# ============================================================
# 15. FUTURE WORKFLOW
# ============================================================

all_future_summary = []
bias_records = []
combined_future_records = []

for scenario, scenario_label in SCENARIOS.items():

    print("\n" + "=" * 80)
    print(f"Processing scenario: {scenario} - {scenario_label}")
    print("=" * 80)

    future_series_list = []

    for model in CLIMATE_MODELS:

        print(f"\nProcessing model: {model}, scenario: {scenario}")

        try:
            future_da = open_future_total_et(
                model_name=model,
                scenario=scenario,
                hist_domain=hist_domain
            )

            future_series_raw = to_monthly_area_series(future_da)
            future_series_raw.name = model

            if APPLY_BIAS_CORRECTION:
                future_series_used, factor, n_overlap = bias_correct_future_series(
                    future_series=future_series_raw,
                    reference_series=historical_df[BIAS_REFERENCE]
                )
            else:
                future_series_used = future_series_raw
                factor = np.nan
                n_overlap = 0

            # Keep only months after historical period for plotting and future analysis
            future_series_plot = future_series_used.loc[
                future_series_used.index >= future_start_date
            ].copy()

            future_series_plot.name = model
            future_series_list.append(future_series_plot)

            bias_records.append({
                "model": model,
                "scenario": scenario,
                "bias_reference": BIAS_REFERENCE,
                "bias_method": BIAS_METHOD if APPLY_BIAS_CORRECTION else "none",
                "bias_factor": factor,
                "n_overlap_months": n_overlap,
            })

            all_future_summary.append({
                "model": model,
                "scenario": scenario,
                "scenario_label": scenario_label,
                "mode": FUTURE_ET_MODE,
                "bias_corrected": APPLY_BIAS_CORRECTION,
                "bias_factor": factor,
                "n_overlap_months": n_overlap,
                "start": future_series_plot.dropna().index.min(),
                "end": future_series_plot.dropna().index.max(),
                "n_months": future_series_plot.dropna().shape[0],
                "mean": future_series_plot.mean(),
                "min": future_series_plot.min(),
                "max": future_series_plot.max(),
                "domain_lat_min": hist_domain[0],
                "domain_lat_max": hist_domain[1],
                "domain_lon_min": hist_domain[2],
                "domain_lon_max": hist_domain[3],
            })

            temp = future_series_plot.to_frame(name="ET")
            temp["model"] = model
            temp["scenario"] = scenario
            temp["scenario_label"] = scenario_label
            temp = temp.reset_index().rename(columns={"index": "time"})
            combined_future_records.append(temp)

            print(
                f"{model} {scenario}: "
                f"bias_factor={factor if np.isfinite(factor) else np.nan:.4f}, "
                f"n_overlap={n_overlap}, "
                f"future_mean={future_series_plot.mean():.4f}"
            )

        except Exception as e:
            print(f"ERROR for {model} / {scenario}: {e}")

    if len(future_series_list) == 0:
        print(f"No valid future data for scenario {scenario}. Plot skipped.")
        continue

    future_df = pd.concat(future_series_list, axis=1)
    future_df = future_df.sort_index()

    # Make sure columns follow the selected model order
    available_cols = [m for m in CLIMATE_MODELS if m in future_df.columns]
    future_df = future_df[available_cols]

    ensemble_mean = future_df.mean(axis=1, skipna=True)
    ensemble_mean.name = "Ensemble_Mean"

    # Save monthly data
    future_csv = os.path.join(
        TABLE_DIR,
        f"future_area_mean_ET_scenario_{scenario}_totalET_only_5models.csv"
    )

    ensemble_csv = os.path.join(
        TABLE_DIR,
        f"future_ensemble_mean_ET_scenario_{scenario}_totalET_only_5models.csv"
    )

    future_df.to_csv(future_csv, encoding="utf-8-sig")
    ensemble_mean.to_frame().to_csv(ensemble_csv, encoding="utf-8-sig")

    # Save annual data
    if SAVE_ANNUAL_SERIES:
        save_series_annual(
            future_df,
            os.path.join(
                TABLE_DIR,
                f"future_area_mean_ET_scenario_{scenario}_annual_totalET_only_5models.csv"
            )
        )

        save_series_annual(
            ensemble_mean.to_frame(),
            os.path.join(
                TABLE_DIR,
                f"future_ensemble_mean_ET_scenario_{scenario}_annual_totalET_only_5models.csv"
            )
        )

    # Plot
    out_png = os.path.join(
        FIG_DIR,
        f"historical_future_ET_scenario_{scenario}_totalET_only_5models.png"
    )

    plot_scenario_timeseries(
        scenario=scenario,
        scenario_label=scenario_label,
        historical_df=historical_df,
        future_df=future_df,
        ensemble_series=ensemble_mean,
        future_period_lines=future_period_lines,
        future_period_labels=future_period_labels,
        out_png=out_png
    )


# ============================================================
# 16. SAVE SUMMARY TABLES
# ============================================================

summary_df = pd.DataFrame(all_future_summary)
summary_csv = os.path.join(TABLE_DIR, "future_models_summary_totalET_only_5models.csv")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

bias_df = pd.DataFrame(bias_records)
bias_csv = os.path.join(TABLE_DIR, "future_bias_correction_factors_totalET_only_5models.csv")
bias_df.to_csv(bias_csv, index=False, encoding="utf-8-sig")

if len(combined_future_records) > 0:
    combined_future_df = pd.concat(combined_future_records, axis=0)

    combined_future_csv = os.path.join(
        TABLE_DIR,
        "future_area_mean_ET_all_models_all_scenarios_totalET_only_5models.csv"
    )

    combined_future_df.to_csv(combined_future_csv, index=False, encoding="utf-8-sig")
else:
    combined_future_csv = None


config_df = pd.DataFrame([
    {"parameter": "FUTURE_ET_MODE", "value": FUTURE_ET_MODE},
    {"parameter": "FUTURE_BASE_DIR", "value": FUTURE_BASE_DIR},
    {"parameter": "CLIMATE_MODELS", "value": ", ".join(CLIMATE_MODELS)},
    {"parameter": "SCENARIOS", "value": ", ".join(SCENARIOS.keys())},
    {"parameter": "APPLY_BIAS_CORRECTION", "value": APPLY_BIAS_CORRECTION},
    {"parameter": "BIAS_REFERENCE", "value": BIAS_REFERENCE},
    {"parameter": "BIAS_METHOD", "value": BIAS_METHOD},
    {"parameter": "ROLLING_MONTHS", "value": ROLLING_MONTHS},
    {"parameter": "future_start_date", "value": future_start_date.strftime("%Y-%m-%d")},
    {"parameter": "future_end_date", "value": future_end_date.strftime("%Y-%m-%d")},
    {"parameter": "domain_lat_min", "value": hist_domain[0]},
    {"parameter": "domain_lat_max", "value": hist_domain[1]},
    {"parameter": "domain_lon_min", "value": hist_domain[2]},
    {"parameter": "domain_lon_max", "value": hist_domain[3]},
])

config_csv = os.path.join(TABLE_DIR, "run_configuration_totalET_only_5models.csv")
config_df.to_csv(config_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 17. FINAL REPORT
# ============================================================

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print("\nMain output folder:")
print(OUT_DIR)

print("\nMain tables:")
print(historical_csv)
print(summary_csv)
print(bias_csv)
print(config_csv)

if combined_future_csv is not None:
    print(combined_future_csv)

print("\nFigures:")
for scenario in SCENARIOS:
    print(
        os.path.join(
            FIG_DIR,
            f"historical_future_ET_scenario_{scenario}_totalET_only_5models.png"
        )
    )

print("\nScientific note:")
print("Future ET is calculated using totalET_monthavg only.")
print("Five future climate models are plotted separately as light lines.")
print("The ensemble mean of the five models is plotted as a bold red line.")
print("Future period starts from the first month after the last historical record.")
print("Bias correction is applied to match future model scale to historical CWatM over the overlap period.")


Loading historical data...

Opening historical dataset: CWatM
E:\ET_Article_With_MM_SA\Future data ST\ET_CWatM_mm_day.nc
Variable: ET_final_mm_day
Dims: ('time', 'lat', 'lon')
Shape: (468, 13, 14)
Time: 1981-01 to 2019-12
Lat: 30.750 to 36.750
Lon: 45.250 to 51.750
Mean: 1.102419

Opening historical dataset: GLDAS
E:\ET_Article_With_MM_SA\Future data ST\GLDAS_ET_2000_2019.nc
Variable: __xarray_dataarray_variable__
Dims: ('time', 'lat', 'lon')
Shape: (240, 13, 14)
Time: 2000-01 to 2019-12
Lat: 30.750 to 36.750
Lon: 45.250 to 51.750
Mean: 0.793585

Opening historical dataset: SMAP
E:\ET_Article_With_MM_SA\Future data ST\SMAP_ET_2000_2019.nc
Variable: __xarray_dataarray_variable__
Dims: ('time', 'lat', 'lon')
Shape: (58, 13, 14)
Time: 2015-03 to 2019-12
Lat: 30.750 to 36.750
Lon: 45.250 to 51.750
Mean: 1.057844

Historical domain intersection:
lat: 30.750 to 36.750
lon: 45.250 to 51.750

Processing future scenario: 126 - Optimistic / SSP126

Processing model: GFDL, scenario: 126

Process

In [8]:
# -*- coding: utf-8 -*-

"""
Historical-Future ET Time Series
Five Climate Models and Three Scenarios

Historical:
    CWatM
    GLDAS
    SMAP

Future climate models:
    GFDL
    IPSL
    MPI
    MRI
    UKESM

Scenarios:
    126
    370
    585

Future mode:
    Conservative mode:
        Future_ET = totalET_monthavg only

Reason:
    totalET_monthavg is already total evapotranspiration.
    EvapoChannel and EvapWaterBodyM are not added in this mode to avoid double counting.

Main outputs:
    Three figures:
        historical_future_ET_scenario_126_totalET_only_5models.png
        historical_future_ET_scenario_370_totalET_only_5models.png
        historical_future_ET_scenario_585_totalET_only_5models.png
"""

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


# ============================================================
# 1. PATH SETTINGS
# ============================================================

# مسیر اصلی همان پوشه‌ای است که فایل‌های nc تاریخی و پوشه‌های GFDL, IPSL, MPI, MRI, UKESM داخل آن هستند.
BASE_DIR = os.getcwd()

# Historical files are directly inside BASE_DIR according to your screenshot
CWATM_FILE = os.path.join(BASE_DIR, "ET_CWatM_mm_day.nc")
GLDAS_FILE = os.path.join(BASE_DIR, "GLDAS_ET_2000_2019.nc")
SMAP_FILE = os.path.join(BASE_DIR, "SMAP_ET_2000_2019.nc")

# Future model folders are also directly inside BASE_DIR
FUTURE_BASE_DIR = BASE_DIR


# ============================================================
# 2. MODELS AND SCENARIOS
# ============================================================

CLIMATE_MODELS = [
    "GFDL",
    "IPSL",
    "MPI",
    "MRI",
    "UKESM",
]

SCENARIOS = {
    "126": "Optimistic / SSP126",
    "370": "Medium / SSP370",
    "585": "Pessimistic / SSP585",
}

FUTURE_TOTAL_ET_FILE = "totalET_monthavg.nc"
FUTURE_TOTAL_ET_VAR = "totalET_monthavg"


# ============================================================
# 3. OUTPUT SETTINGS
# ============================================================

OUT_DIR = os.path.join(BASE_DIR, "outputs_historical_future_ET_totalET_only_5models")
TABLE_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figures")

for d in [OUT_DIR, TABLE_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)


# ============================================================
# 4. ANALYSIS SETTINGS
# ============================================================

FUTURE_ET_MODE = "totalET_only"

# Historical products are assumed to be monthly mean daily ET, mm/day
HISTORICAL_UNIT = "mm/day"

# Future totalET_monthavg has unit m in the files.
# Multiplying by 1000 gives mm-scale.
FUTURE_CONVERT_M_TO_MM = True

MASK_NEGATIVE_HISTORICAL_ET = True
MASK_NEGATIVE_FUTURE_ET = True

# Bias correction:
# True  = future models are adjusted to CWatM over the overlap period, usually 2015-2019.
# False = raw future totalET values are plotted.
APPLY_BIAS_CORRECTION = True
BIAS_REFERENCE = "CWatM"

# Multiplicative correction:
# corrected_future = raw_future * mean(CWatM_overlap) / mean(future_overlap)
BIAS_METHOD = "multiplicative_mean_ratio"

# 1  = raw monthly series
# 12 = 12-month rolling mean for smoother plots
ROLLING_MONTHS = 12

SAVE_ANNUAL_SERIES = True


# ============================================================
# 5. PLOT SETTINGS
# ============================================================

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 150


# ============================================================
# 6. BASIC FUNCTIONS
# ============================================================

def ensure_month_start_index(series_or_df):
    obj = series_or_df.copy()
    obj.index = pd.to_datetime(obj.index)
    obj.index = obj.index.to_period("M").to_timestamp()
    obj = obj.sort_index()
    return obj


def apply_rolling(series, months):
    if months is None or months <= 1:
        return series

    return series.rolling(
        window=months,
        center=True,
        min_periods=max(2, months // 2)
    ).mean()


def get_y_label():
    if APPLY_BIAS_CORRECTION:
        return "ET (bias-corrected; comparable historical-future scale)"
    return "ET (historical: mm/day; future totalET converted from m to mm)"


def save_series_annual(df, out_csv):
    annual_df = df.copy()
    annual_df.index = pd.to_datetime(annual_df.index)
    annual_df = annual_df.resample("YE").mean()
    annual_df.index = annual_df.index.year
    annual_df.index.name = "Year"
    annual_df.to_csv(out_csv, encoding="utf-8-sig")
    return annual_df


# ============================================================
# 7. HISTORICAL DATA READER
# ============================================================

def extract_dates_from_long_name(long_name_attr):
    """
    Extract dates from GLDAS/SMAP long_name attributes:
        0_Evap_tavg_2000_01
        57_land_evapotranspiration_flux_2019_12
    """

    if isinstance(long_name_attr, str):
        long_names = [long_name_attr]
    else:
        long_names = list(long_name_attr)

    dates = []

    for item in long_names:
        item = str(item)
        match = re.search(r"(\d{4})_(\d{2})", item)

        if match is None:
            raise ValueError(f"Cannot extract date from long_name item: {item}")

        dates.append(
            pd.Timestamp(
                year=int(match.group(1)),
                month=int(match.group(2)),
                day=1
            )
        )

    return pd.DatetimeIndex(dates)


def standardize_xy_to_latlon(da):
    """
    Rename coordinates:
        x / longitude -> lon
        y / latitude  -> lat
    """

    rename_dict = {}

    for name in list(da.dims) + list(da.coords):
        low = str(name).lower()

        if low in ["x", "longitude"]:
            rename_dict[name] = "lon"

        elif low in ["y", "latitude"]:
            rename_dict[name] = "lat"

    if rename_dict:
        da = da.rename(rename_dict)

    if "lat" in da.coords:
        da = da.sortby("lat")

    if "lon" in da.coords:
        da = da.sortby("lon")

    return da


def open_historical_et(path, dataset_name):
    print("\n" + "=" * 80)
    print(f"Opening historical dataset: {dataset_name}")
    print(path)
    print("=" * 80)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    ds = xr.open_dataset(path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if len(candidate_vars) == 0:
        raise ValueError(f"No variable found in {dataset_name}")

    if "ET_final_mm_day" in candidate_vars:
        var_name = "ET_final_mm_day"
    elif "__xarray_dataarray_variable__" in candidate_vars:
        var_name = "__xarray_dataarray_variable__"
    else:
        var_name = candidate_vars[0]

    da = ds[var_name]

    if "time" in da.dims:
        da["time"] = pd.to_datetime(da.time.values)

    elif "band" in da.dims:
        long_name = da.attrs.get("long_name", None)

        if long_name is None:
            raise ValueError(f"{dataset_name} has band dimension but no long_name.")

        dates = extract_dates_from_long_name(long_name)

        if len(dates) != da.sizes["band"]:
            raise ValueError(
                f"{dataset_name}: extracted dates do not match band size."
            )

        da = da.rename({"band": "time"})
        da = da.assign_coords(time=dates)

    else:
        raise ValueError(f"No time or band dimension found in {dataset_name}")

    da = standardize_xy_to_latlon(da)

    for d in ["time", "lat", "lon"]:
        if d not in da.dims:
            raise ValueError(f"{dataset_name}: required dimension '{d}' not found.")

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    if MASK_NEGATIVE_HISTORICAL_ET:
        da = da.where(da >= 0)

    da.name = dataset_name
    da.attrs["unit"] = HISTORICAL_UNIT

    print(f"Variable: {var_name}")
    print(f"Dims: {da.dims}")
    print(f"Shape: {da.shape}")
    print(
        "Time:",
        pd.to_datetime(da.time.values[0]).strftime("%Y-%m"),
        "to",
        pd.to_datetime(da.time.values[-1]).strftime("%Y-%m")
    )
    print(f"Lat: {float(da.lat.min()):.3f} to {float(da.lat.max()):.3f}")
    print(f"Lon: {float(da.lon.min()):.3f} to {float(da.lon.max()):.3f}")
    print(f"Mean: {float(da.mean(skipna=True)):.6f}")

    return da


# ============================================================
# 8. CELL AREA AND AREA-WEIGHTED MEAN
# ============================================================

def infer_bounds_from_centers(values):
    values = np.asarray(values, dtype=float)

    if len(values) < 2:
        raise ValueError("At least two coordinate values are needed.")

    values_sorted = np.sort(values)

    mid = (values_sorted[:-1] + values_sorted[1:]) / 2.0

    first = values_sorted[0] - (mid[0] - values_sorted[0])
    last = values_sorted[-1] + (values_sorted[-1] - mid[-1])

    return np.concatenate([[first], mid, [last]])


def calculate_latlon_cell_area(lat, lon, radius=6371000.0):
    """
    Calculate grid-cell area in m2 for regular lat-lon grid.
    """

    lat_values = np.asarray(lat.values if hasattr(lat, "values") else lat, dtype=float)
    lon_values = np.asarray(lon.values if hasattr(lon, "values") else lon, dtype=float)

    lat_ascending = np.all(np.diff(lat_values) > 0)
    lon_ascending = np.all(np.diff(lon_values) > 0)

    lat_sorted = np.sort(lat_values)
    lon_sorted = np.sort(lon_values)

    lat_bounds = infer_bounds_from_centers(lat_sorted)
    lon_bounds = infer_bounds_from_centers(lon_sorted)

    lat_bounds_rad = np.deg2rad(lat_bounds)
    lon_bounds_rad = np.deg2rad(lon_bounds)

    dlon = np.diff(lon_bounds_rad)
    sin_lat_diff = np.diff(np.sin(lat_bounds_rad))

    area_sorted = (radius ** 2) * sin_lat_diff[:, None] * dlon[None, :]
    area_sorted = np.abs(area_sorted)

    if not lat_ascending:
        area_sorted = area_sorted[::-1, :]

    if not lon_ascending:
        area_sorted = area_sorted[:, ::-1]

    area = xr.DataArray(
        area_sorted,
        dims=("lat", "lon"),
        coords={"lat": lat_values, "lon": lon_values},
        name="cell_area"
    )

    area.attrs["units"] = "m2"

    return area


def area_weighted_mean(da):
    """
    Area-weighted spatial mean over lat/lon.
    """

    area = calculate_latlon_cell_area(da.lat, da.lon)
    valid = da.notnull()

    weighted_sum = (da * area).where(valid).sum(dim=("lat", "lon"), skipna=True)
    weight_sum = area.where(valid).sum(dim=("lat", "lon"), skipna=True)

    out = weighted_sum / weight_sum
    out.name = da.name

    return out


def to_monthly_area_series(da):
    s = area_weighted_mean(da).to_series()
    s = ensure_month_start_index(s)
    return s


# ============================================================
# 9. DOMAIN FUNCTIONS
# ============================================================

def get_historical_domain(*arrays):
    """
    Use intersection of historical product bounding boxes.
    """

    lat_min = max([float(da.lat.min()) for da in arrays])
    lat_max = min([float(da.lat.max()) for da in arrays])
    lon_min = max([float(da.lon.min()) for da in arrays])
    lon_max = min([float(da.lon.max()) for da in arrays])

    return lat_min, lat_max, lon_min, lon_max


def clip_to_domain(da, domain):
    lat_min, lat_max, lon_min, lon_max = domain

    da = da.sortby("lat")
    da = da.sortby("lon")

    clipped = da.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )

    return clipped


# ============================================================
# 10. FUTURE TOTAL ET READER
# ============================================================

def open_future_total_et(model_name, scenario, hist_domain):
    """
    Open future totalET_monthavg.nc for one model and one scenario.
    Conservative mode:
        Future_ET = totalET_monthavg only
    """

    scen_dir = os.path.join(FUTURE_BASE_DIR, model_name, scenario)
    total_path = os.path.join(scen_dir, FUTURE_TOTAL_ET_FILE)

    if not os.path.exists(total_path):
        raise FileNotFoundError(total_path)

    ds = xr.open_dataset(total_path)

    candidate_vars = [
        v for v in ds.data_vars
        if str(v).lower() not in ["spatial_ref", "crs"]
    ]

    if FUTURE_TOTAL_ET_VAR in candidate_vars:
        var_name = FUTURE_TOTAL_ET_VAR
    elif len(candidate_vars) == 1:
        var_name = candidate_vars[0]
    else:
        raise ValueError(
            f"Cannot select future total ET variable from {total_path}. "
            f"Candidates: {candidate_vars}"
        )

    da = ds[var_name]

    if "time" not in da.dims:
        raise ValueError(f"No time dimension found in {total_path}")

    da["time"] = pd.to_datetime(da.time.values)

    da = standardize_xy_to_latlon(da)

    for d in ["time", "lat", "lon"]:
        if d not in da.dims:
            raise ValueError(f"{model_name}/{scenario}: dimension '{d}' not found.")

    da = da.astype("float64")
    da = da.where(np.isfinite(da))

    da = clip_to_domain(da, hist_domain)

    if FUTURE_CONVERT_M_TO_MM:
        da = da * 1000.0

    if MASK_NEGATIVE_FUTURE_ET:
        da = da.where(da >= 0)

    da.name = f"{model_name}_{scenario}_totalET"

    return da


# ============================================================
# 11. BIAS CORRECTION
# ============================================================

def bias_correct_future_series(future_series, reference_series):
    """
    Multiplicative mean-ratio correction using overlap period.

    corrected = future * mean(reference_overlap) / mean(future_overlap)
    """

    df = pd.concat([future_series, reference_series], axis=1).dropna()
    df.columns = ["future", "reference"]

    n_overlap = len(df)

    if n_overlap < 12:
        return future_series, np.nan, n_overlap

    future_mean = df["future"].mean()
    reference_mean = df["reference"].mean()

    if not np.isfinite(future_mean) or future_mean == 0:
        return future_series, np.nan, n_overlap

    factor = reference_mean / future_mean

    corrected = future_series * factor
    corrected.name = future_series.name

    return corrected, factor, n_overlap


# ============================================================
# 12. FUTURE PERIODS
# ============================================================

def get_future_start_date_from_historical(historical_df):
    """
    Future starts from first month after the last available historical month.
    """

    last_hist_date = historical_df.dropna(how="all").index.max()
    future_start = pd.Timestamp(last_hist_date) + pd.DateOffset(months=1)
    future_start = future_start.to_period("M").to_timestamp()

    return future_start


def make_future_period_lines_and_labels(future_start, future_end):
    """
    Three 25-year future periods starting from future_start.
    """

    start1 = pd.Timestamp(future_start)
    start2 = start1 + pd.DateOffset(years=25)
    start3 = start2 + pd.DateOffset(years=25)
    start4 = start3 + pd.DateOffset(years=25)

    future_end = pd.Timestamp(future_end)

    if start4 > future_end:
        start4 = future_end

    lines = [start1, start2, start3, start4]

    labels = [
        ("Near future", start1, start2 - pd.DateOffset(months=1)),
        ("Mid-century", start2, start3 - pd.DateOffset(months=1)),
        ("Late century", start3, future_end),
    ]

    return lines, labels


# ============================================================
# 13. PLOTTING
# ============================================================

def plot_scenario_timeseries(
    scenario,
    scenario_label,
    historical_df,
    future_df,
    ensemble_series,
    future_period_lines,
    future_period_labels,
    out_png
):
    """
    Plot one scenario:
        Historical: CWatM, GLDAS, SMAP
        Future: 5 individual model lines + ensemble mean
    """

    plt.figure(figsize=(16, 6.5))

    hist_colors = {
        "CWatM": "black",
        "GLDAS": "tab:blue",
        "SMAP": "tab:green",
    }

    # Historical lines
    for col in historical_df.columns:
        s = apply_rolling(historical_df[col], ROLLING_MONTHS)

        plt.plot(
            s.index,
            s.values,
            linewidth=2.1,
            label=f"Historical {col}",
            color=hist_colors.get(col, None),
            zorder=3
        )

    # Future individual model lines
    model_colors = {
        "GFDL": "tab:blue",
        "IPSL": "tab:orange",
        "MPI": "tab:green",
        "MRI": "tab:purple",
        "UKESM": "tab:brown",
    }

    for col in future_df.columns:
        s = apply_rolling(future_df[col], ROLLING_MONTHS)

        plt.plot(
            s.index,
            s.values,
            linewidth=1.5,
            alpha=0.45,
            color=model_colors.get(col, None),
            label=f"Future {col}",
            zorder=4
        )

    # Ensemble mean
    ens = apply_rolling(ensemble_series, ROLLING_MONTHS)

    plt.plot(
        ens.index,
        ens.values,
        linewidth=3.4,
        color="red",
        alpha=0.95,
        label="Future ensemble mean",
        zorder=5
    )

    # Period lines
    for dt in future_period_lines:
        plt.axvline(
            pd.Timestamp(dt),
            color="gray",
            linestyle="--",
            linewidth=1.0,
            alpha=0.45,
            zorder=1
        )

    ymin, ymax = plt.ylim()

    # Period labels
    for label, start, end in future_period_labels:
        mid = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2

        plt.text(
            mid,
            ymax - 0.03 * (ymax - ymin),
            label,
            ha="center",
            va="top",
            fontsize=9,
            color="gray"
        )

    title_extra = "bias corrected to historical CWatM" if APPLY_BIAS_CORRECTION else "raw future"

    plt.title(
        f"Historical and Future Monthly ET Time Series - {scenario_label}\n"
        f"Future mode: totalET only; {title_extra}"
    )

    plt.xlabel("Time")
    plt.ylabel(get_y_label())
    plt.grid(True, alpha=0.25)
    plt.legend(ncol=3, fontsize=8)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


# ============================================================
# 14. MAIN WORKFLOW
# ============================================================

print("\nLoading historical ET datasets...")

cwatm = open_historical_et(CWATM_FILE, "CWatM")
gldas = open_historical_et(GLDAS_FILE, "GLDAS")
smap = open_historical_et(SMAP_FILE, "SMAP")

hist_domain = get_historical_domain(cwatm, gldas, smap)

print("\nHistorical domain intersection:")
print(f"lat: {hist_domain[0]:.3f} to {hist_domain[1]:.3f}")
print(f"lon: {hist_domain[2]:.3f} to {hist_domain[3]:.3f}")

# Clip historical data to common historical domain
cwatm_clip = clip_to_domain(cwatm, hist_domain)
gldas_clip = clip_to_domain(gldas, hist_domain)
smap_clip = clip_to_domain(smap, hist_domain)

historical_df = pd.concat(
    [
        to_monthly_area_series(cwatm_clip).rename("CWatM"),
        to_monthly_area_series(gldas_clip).rename("GLDAS"),
        to_monthly_area_series(smap_clip).rename("SMAP"),
    ],
    axis=1
)

historical_df = historical_df.sort_index()

historical_csv = os.path.join(TABLE_DIR, "historical_area_mean_ET.csv")
historical_df.to_csv(historical_csv, encoding="utf-8-sig")

if SAVE_ANNUAL_SERIES:
    save_series_annual(
        historical_df,
        os.path.join(TABLE_DIR, "historical_area_mean_ET_annual.csv")
    )

future_start_date = get_future_start_date_from_historical(historical_df)
future_end_date = pd.Timestamp("2100-12-01")

future_period_lines, future_period_labels = make_future_period_lines_and_labels(
    future_start=future_start_date,
    future_end=future_end_date
)

print("\nFuture plotting starts from:")
print(future_start_date.strftime("%Y-%m"))

print("\nClimate models:")
print(CLIMATE_MODELS)


# ============================================================
# 15. FUTURE WORKFLOW
# ============================================================

all_future_summary = []
bias_records = []
combined_future_records = []

for scenario, scenario_label in SCENARIOS.items():

    print("\n" + "=" * 80)
    print(f"Processing scenario: {scenario} - {scenario_label}")
    print("=" * 80)

    future_series_list = []

    for model in CLIMATE_MODELS:

        print(f"\nProcessing model: {model}, scenario: {scenario}")

        try:
            future_da = open_future_total_et(
                model_name=model,
                scenario=scenario,
                hist_domain=hist_domain
            )

            future_series_raw = to_monthly_area_series(future_da)
            future_series_raw.name = model

            if APPLY_BIAS_CORRECTION:
                future_series_used, factor, n_overlap = bias_correct_future_series(
                    future_series=future_series_raw,
                    reference_series=historical_df[BIAS_REFERENCE]
                )
            else:
                future_series_used = future_series_raw
                factor = np.nan
                n_overlap = 0

            # Keep only months after historical period for plotting and future analysis
            future_series_plot = future_series_used.loc[
                future_series_used.index >= future_start_date
            ].copy()

            future_series_plot.name = model
            future_series_list.append(future_series_plot)

            bias_records.append({
                "model": model,
                "scenario": scenario,
                "bias_reference": BIAS_REFERENCE,
                "bias_method": BIAS_METHOD if APPLY_BIAS_CORRECTION else "none",
                "bias_factor": factor,
                "n_overlap_months": n_overlap,
            })

            all_future_summary.append({
                "model": model,
                "scenario": scenario,
                "scenario_label": scenario_label,
                "mode": FUTURE_ET_MODE,
                "bias_corrected": APPLY_BIAS_CORRECTION,
                "bias_factor": factor,
                "n_overlap_months": n_overlap,
                "start": future_series_plot.dropna().index.min(),
                "end": future_series_plot.dropna().index.max(),
                "n_months": future_series_plot.dropna().shape[0],
                "mean": future_series_plot.mean(),
                "min": future_series_plot.min(),
                "max": future_series_plot.max(),
                "domain_lat_min": hist_domain[0],
                "domain_lat_max": hist_domain[1],
                "domain_lon_min": hist_domain[2],
                "domain_lon_max": hist_domain[3],
            })

            temp = future_series_plot.to_frame(name="ET")
            temp["model"] = model
            temp["scenario"] = scenario
            temp["scenario_label"] = scenario_label
            temp = temp.reset_index().rename(columns={"index": "time"})
            combined_future_records.append(temp)

            print(
                f"{model} {scenario}: "
                f"bias_factor={factor if np.isfinite(factor) else np.nan:.4f}, "
                f"n_overlap={n_overlap}, "
                f"future_mean={future_series_plot.mean():.4f}"
            )

        except Exception as e:
            print(f"ERROR for {model} / {scenario}: {e}")

    if len(future_series_list) == 0:
        print(f"No valid future data for scenario {scenario}. Plot skipped.")
        continue

    future_df = pd.concat(future_series_list, axis=1)
    future_df = future_df.sort_index()

    # Make sure columns follow the selected model order
    available_cols = [m for m in CLIMATE_MODELS if m in future_df.columns]
    future_df = future_df[available_cols]

    ensemble_mean = future_df.mean(axis=1, skipna=True)
    ensemble_mean.name = "Ensemble_Mean"

    # Save monthly data
    future_csv = os.path.join(
        TABLE_DIR,
        f"future_area_mean_ET_scenario_{scenario}_totalET_only_5models.csv"
    )

    ensemble_csv = os.path.join(
        TABLE_DIR,
        f"future_ensemble_mean_ET_scenario_{scenario}_totalET_only_5models.csv"
    )

    future_df.to_csv(future_csv, encoding="utf-8-sig")
    ensemble_mean.to_frame().to_csv(ensemble_csv, encoding="utf-8-sig")

    # Save annual data
    if SAVE_ANNUAL_SERIES:
        save_series_annual(
            future_df,
            os.path.join(
                TABLE_DIR,
                f"future_area_mean_ET_scenario_{scenario}_annual_totalET_only_5models.csv"
            )
        )

        save_series_annual(
            ensemble_mean.to_frame(),
            os.path.join(
                TABLE_DIR,
                f"future_ensemble_mean_ET_scenario_{scenario}_annual_totalET_only_5models.csv"
            )
        )

    # Plot
    out_png = os.path.join(
        FIG_DIR,
        f"historical_future_ET_scenario_{scenario}_totalET_only_5models.png"
    )

    plot_scenario_timeseries(
        scenario=scenario,
        scenario_label=scenario_label,
        historical_df=historical_df,
        future_df=future_df,
        ensemble_series=ensemble_mean,
        future_period_lines=future_period_lines,
        future_period_labels=future_period_labels,
        out_png=out_png
    )


# ============================================================
# 16. SAVE SUMMARY TABLES
# ============================================================

summary_df = pd.DataFrame(all_future_summary)
summary_csv = os.path.join(TABLE_DIR, "future_models_summary_totalET_only_5models.csv")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

bias_df = pd.DataFrame(bias_records)
bias_csv = os.path.join(TABLE_DIR, "future_bias_correction_factors_totalET_only_5models.csv")
bias_df.to_csv(bias_csv, index=False, encoding="utf-8-sig")

if len(combined_future_records) > 0:
    combined_future_df = pd.concat(combined_future_records, axis=0)

    combined_future_csv = os.path.join(
        TABLE_DIR,
        "future_area_mean_ET_all_models_all_scenarios_totalET_only_5models.csv"
    )

    combined_future_df.to_csv(combined_future_csv, index=False, encoding="utf-8-sig")
else:
    combined_future_csv = None


config_df = pd.DataFrame([
    {"parameter": "FUTURE_ET_MODE", "value": FUTURE_ET_MODE},
    {"parameter": "FUTURE_BASE_DIR", "value": FUTURE_BASE_DIR},
    {"parameter": "CLIMATE_MODELS", "value": ", ".join(CLIMATE_MODELS)},
    {"parameter": "SCENARIOS", "value": ", ".join(SCENARIOS.keys())},
    {"parameter": "APPLY_BIAS_CORRECTION", "value": APPLY_BIAS_CORRECTION},
    {"parameter": "BIAS_REFERENCE", "value": BIAS_REFERENCE},
    {"parameter": "BIAS_METHOD", "value": BIAS_METHOD},
    {"parameter": "ROLLING_MONTHS", "value": ROLLING_MONTHS},
    {"parameter": "future_start_date", "value": future_start_date.strftime("%Y-%m-%d")},
    {"parameter": "future_end_date", "value": future_end_date.strftime("%Y-%m-%d")},
    {"parameter": "domain_lat_min", "value": hist_domain[0]},
    {"parameter": "domain_lat_max", "value": hist_domain[1]},
    {"parameter": "domain_lon_min", "value": hist_domain[2]},
    {"parameter": "domain_lon_max", "value": hist_domain[3]},
])

config_csv = os.path.join(TABLE_DIR, "run_configuration_totalET_only_5models.csv")
config_df.to_csv(config_csv, index=False, encoding="utf-8-sig")


# ============================================================
# 17. FINAL REPORT
# ============================================================

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print("\nMain output folder:")
print(OUT_DIR)

print("\nMain tables:")
print(historical_csv)
print(summary_csv)
print(bias_csv)
print(config_csv)

if combined_future_csv is not None:
    print(combined_future_csv)

print("\nFigures:")
for scenario in SCENARIOS:
    print(
        os.path.join(
            FIG_DIR,
            f"historical_future_ET_scenario_{scenario}_totalET_only_5models.png"
        )
    )

print("\nScientific note:")
print("Future ET is calculated using totalET_monthavg only.")
print("Five future climate models are plotted separately as light lines.")
print("The ensemble mean of the five models is plotted as a bold red line.")
print("Future period starts from the first month after the last historical record.")
print("Bias correction is applied to match future model scale to historical CWatM over the overlap period.")


Loading historical ET datasets...

Opening historical dataset: CWatM
E:\ET_Article_With_MM_SA\Future data ST\ET_CWatM_mm_day.nc
Variable: ET_final_mm_day
Dims: ('time', 'lat', 'lon')
Shape: (468, 13, 14)
Time: 1981-01 to 2019-12
Lat: 30.750 to 36.750
Lon: 45.250 to 51.750
Mean: 1.102419

Opening historical dataset: GLDAS
E:\ET_Article_With_MM_SA\Future data ST\GLDAS_ET_2000_2019.nc
Variable: __xarray_dataarray_variable__
Dims: ('time', 'lat', 'lon')
Shape: (240, 13, 14)
Time: 2000-01 to 2019-12
Lat: 30.750 to 36.750
Lon: 45.250 to 51.750
Mean: 0.793585

Opening historical dataset: SMAP
E:\ET_Article_With_MM_SA\Future data ST\SMAP_ET_2000_2019.nc
Variable: __xarray_dataarray_variable__
Dims: ('time', 'lat', 'lon')
Shape: (58, 13, 14)
Time: 2015-03 to 2019-12
Lat: 30.750 to 36.750
Lon: 45.250 to 51.750
Mean: 1.057844

Historical domain intersection:
lat: 30.750 to 36.750
lon: 45.250 to 51.750

Future plotting starts from:
2020-01

Climate models:
['GFDL', 'IPSL', 'MPI', 'MRI', 'UKESM']
